# E3-TSF Final Kaggle Run (fixed data discovery)

Run the cells in order.

- **Cell 1 - DATA CHECK**: lists what is actually mounted under `/kaggle/input`.
  If it reports 0 files, attach the datasets and use *Run -> Restart & clear cell
  outputs* before doing anything else. The main cell cannot work without this.
- **Cell 2 - MAIN**: the full experiment. It now finds the CSVs wherever they sit
  under `/kaggle/input` (any folder name, any depth, also inside `.zip`), matches
  them by filename and by column schema, and raises a clear error instead of
  silently skipping every dataset.

Outputs are written to `/kaggle/working/e3_tsf_outputs_paper_final`:

- `tables/`: metrics, predictions, gate weights, input-length experiment CSVs
- `paper_figures/`: publication figures
- `figure_source_data/`: the numbers behind each figure
- `checkpoints/`, `metadata/`

If a dataset lives somewhere unusual, add its folder to `EXTRA_DATA_DIRS` in the
main cell. With *Settings -> Internet = ON* the notebook can also download the
public datasets itself via `kagglehub` (`TRY_KAGGLEHUB_FALLBACK = True`).


In [ ]:
# ============================================================
# CELL 1 - DATA CHECK. Run this first, before the main cell.
# It only inspects the filesystem, so it takes a second.
# ============================================================
import os
import sys

print("python:", sys.version.split()[0])
print("cwd   :", os.getcwd())

for root in ["/kaggle", "/kaggle/input", "/kaggle/working"]:
    print(f"{root:16s} exists: {os.path.isdir(root)}")

root = "/kaggle/input"
if not os.path.isdir(root):
    print("\nThis is not a Kaggle session (no /kaggle/input).")
else:
    datasets = sorted(d for d in os.listdir(root) if os.path.isdir(os.path.join(root, d)))
    print(f"\nAttached input datasets ({len(datasets)}):")
    for d in datasets:
        print("  -", d)

    n_files = 0
    print("\nAll files under /kaggle/input:")
    for dirpath, dirnames, filenames in os.walk(root):
        for filename in sorted(filenames):
            path = os.path.join(dirpath, filename)
            n_files += 1
            if n_files <= 200:
                try:
                    print(f"  {path}   ({os.path.getsize(path)/1e6:.2f} MB)")
                except OSError:
                    print(f"  {path}")
    if n_files > 200:
        print(f"  ... {n_files - 200} more files")
    print(f"\nTotal files under /kaggle/input: {n_files}")

    if n_files == 0:
        print(
            "\n" + "!" * 78 + "\n"
            "Nothing is mounted. The datasets are NOT attached to this session.\n"
            "Do this in the Kaggle editor:\n"
            "  1. Right sidebar -> Input -> '+ Add Input' -> add each dataset.\n"
            "  2. Run -> 'Restart & clear cell outputs'  (mandatory: a session\n"
            "     that was already running does not always see new inputs).\n"
            "  3. Re-run this cell. You must see the CSV files listed above\n"
            "     before running the main cell.\n"
            "Note: if you are using 'Save Version -> Save & Run All', the inputs\n"
            "must be attached in the editor and the notebook saved afterwards.\n"
            + "!" * 78
        )
    else:
        print("\nData is visible. Copy the paths above if you need to edit DATASETS.")


In [ ]:
"""
E3-TSF: Dynamic Gated Multi-Expert Transformer Framework
for Robust Multi-Domain Time Series Forecasting

Fresh multi-dataset executable notebook with publication-ready figures.

Datasets included by default:
1) Daily Delhi Climate
2) Demand forecasting panel dataset: store-item sales
3) PJMW hourly energy load
4) Traffic volume hourly dataset

Major improvements compared with the earlier two-dataset script:
- Includes the four datasets used in the manuscript.
- Automatically resolves Kaggle input paths when dataset folder names differ.
- Correct panel-series processing for Demand using store/item series IDs.
- Train/validation/test chronological split.
- Scaling fitted only on training windows/targets.
- Wavelet decomposition is built window-by-window to avoid future leakage.
- E3-TSF dynamic gated multi-expert model.
- Panel-aware optional series ID embedding.
- Improved training: AdamW, warmup + cosine decay, mixed precision on CUDA,
  robust loss, early stopping, gradient clipping, gate entropy regularization,
  expert diversity regularization, RevIN-style instance normalization,
  learnable-temperature sparse gating, and multi-seed mean/std reporting.
- Baselines and ablations:
    Seasonal Naive
    DLinear
    PatchTST-like Transformer
    E3TSF_NoGateAvg
    E3TSF_DynamicGated
- Saves dataset-wise results, prediction files, gate weights, plots, and a final summary.

How to run on Kaggle:
    python e3_tsf_all_datasets_advanced_training.py

Edit the DATASETS section if Kaggle input paths differ.
"""

import os
import sys
import math
import json
import random
import time
import copy
import pickle
import platform
import warnings
from dataclasses import dataclass, asdict
from typing import Dict, List, Optional, Tuple

warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import sklearn

try:
    import pywt
    HAS_PYWT = True
except ImportError:
    pywt = None
    HAS_PYWT = False
    print("PyWavelets is not installed; using a moving-average multiscale fallback.")

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader

from sklearn.preprocessing import StandardScaler
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

try:
    import matplotlib.pyplot as plt
    HAS_MPL = True
except Exception:
    HAS_MPL = False


# ============================================================
# 1. Global configuration
# ============================================================

SEED = 42
# Final paper run. Use all three seeds for the tables that report mean/std.
RUN_SEEDS = [42, 123, 2026]
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
OUT_DIR = "/kaggle/working/e3_tsf_outputs_paper_final" if os.path.exists("/kaggle") else "./e3_tsf_outputs_paper_final"
PAPER_FIG_DIR = os.path.join(OUT_DIR, "paper_figures")
FIGURE_SOURCE_DIR = os.path.join(OUT_DIR, "figure_source_data")
TABLE_DIR = os.path.join(OUT_DIR, "tables")
CHECKPOINT_DIR = os.path.join(OUT_DIR, "checkpoints")
METADATA_DIR = os.path.join(OUT_DIR, "metadata")
os.makedirs(OUT_DIR, exist_ok=True)
for _d in [PAPER_FIG_DIR, FIGURE_SOURCE_DIR, TABLE_DIR, CHECKPOINT_DIR, METADATA_DIR]:
    os.makedirs(_d, exist_ok=True)


def set_seed(seed: int = 42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False
    try:
        torch.use_deterministic_algorithms(True, warn_only=True)
    except Exception:
        pass


def sync_device():
    if torch.cuda.is_available():
        torch.cuda.synchronize()


def make_amp_scaler(enabled: bool):
    if hasattr(torch, "amp") and hasattr(torch.amp, "GradScaler"):
        try:
            return torch.amp.GradScaler("cuda", enabled=enabled)
        except TypeError:
            return torch.amp.GradScaler(enabled=enabled)
    return torch.cuda.amp.GradScaler(enabled=enabled)


def amp_autocast(enabled: bool):
    if hasattr(torch, "amp") and hasattr(torch.amp, "autocast"):
        try:
            return torch.amp.autocast("cuda", enabled=enabled)
        except TypeError:
            return torch.amp.autocast(device_type="cuda", enabled=enabled)
    return torch.cuda.amp.autocast(enabled=enabled)


set_seed(SEED)


@dataclass
class ModelConfig:
    lookback: int
    horizon: int
    patch_len: int
    seasonal_period: int
    d_model: int = 128
    n_heads: int = 4
    enc_layers: int = 2
    ff_dim: int = 256
    dropout: float = 0.15
    wavelet: str = "db4"
    wavelet_levels: int = 3
    batch_size: int = 128
    epochs: int = 60
    early_stop: int = 10
    lr: float = 1e-3
    min_lr: float = 1e-5
    weight_decay: float = 1e-4
    grad_clip: float = 1.0
    num_workers: int = 0
    stride: int = 1
    max_series: Optional[int] = None
    max_train_windows: Optional[int] = None
    max_val_windows: Optional[int] = None
    max_test_windows: Optional[int] = None
    gate_temperature: float = 1.0
    gate_entropy_lambda: float = 1e-4
    gate_diversity_lambda: float = 1e-4
    gate_balance_lambda: float = 5e-5
    use_learnable_temperature: bool = True
    sparse_top_k: int = 2              # 0/None disables sparse gating; 2 keeps strongest two experts
    use_revin: bool = True             # RevIN-style instance normalization inside E3-TSF only
    use_amp: bool = True


DATASETS = [
    {
        "name": "DailyClimate",
        "train_path": "/kaggle/input/dailyclimate/DailyDelhiClimateTrain.csv",
        "test_path": "/kaggle/input/dailyclimate/DailyDelhiClimateTest.csv",
        "time_col": "date",
        "target_col": "meantemp",
        "freq": "D",
        "id_cols": None,
        "config": ModelConfig(
            lookback=180, horizon=7, patch_len=12, seasonal_period=7,
            d_model=128, n_heads=4, enc_layers=2, ff_dim=256,
            dropout=0.15, wavelet_levels=3, batch_size=64,
            epochs=90, early_stop=15, lr=8e-4, stride=1,
            gate_entropy_lambda=1e-4,
        ),
    },
    {
        "name": "Demand",
        "train_path": "/kaggle/input/demanddataset/train.csv",
        "test_path": "/kaggle/input/demanddataset/test.csv",
        "time_col": "date",
        "target_col": "sales",
        "freq": "D",
        "id_cols": ["store", "item"],
        "config": ModelConfig(
            lookback=180, horizon=7, patch_len=12, seasonal_period=7,
            d_model=128, n_heads=4, enc_layers=2, ff_dim=256,
            dropout=0.12, wavelet_levels=3, batch_size=256,
            epochs=60, early_stop=10, lr=1e-3, stride=7,
            max_series=None,
            max_train_windows=None,
            max_val_windows=None,
            max_test_windows=None,
            gate_entropy_lambda=2e-4,
        ),
    },
    {
        "name": "PJMW_Hourly",
        "single_path": "/kaggle/input/timeseriesdataset/PJMW_hourly.csv",
        "time_col": "Datetime",
        "target_col": "PJMW_MW",
        "freq": "H",
        "id_cols": None,
        "config": ModelConfig(
            lookback=336, horizon=24, patch_len=24, seasonal_period=24,
            d_model=128, n_heads=4, enc_layers=2, ff_dim=256,
            dropout=0.12, wavelet_levels=3, batch_size=128,
            epochs=70, early_stop=12, lr=8e-4, stride=1,
            max_train_windows=None,
            max_val_windows=None,
            max_test_windows=None,
            gate_entropy_lambda=2e-4,
        ),
    },
    {
        "name": "Traffic",
        "single_path": "/kaggle/input/timeseriesdataset/Traffic.csv",
        "time_col": "date_time",
        "target_col": "traffic_volume",
        "freq": "H",
        "id_cols": None,
        "config": ModelConfig(
            lookback=336, horizon=24, patch_len=24, seasonal_period=24,
            d_model=128, n_heads=4, enc_layers=2, ff_dim=256,
            dropout=0.18, wavelet_levels=3, batch_size=128,
            epochs=80, early_stop=14, lr=7e-4, stride=1,
            gate_entropy_lambda=2e-4,
        ),
    },
]


# ============================================================
# 2. Metrics and utilities
# ============================================================

def compute_metrics(y_true: np.ndarray, y_pred: np.ndarray) -> Dict[str, float]:
    y_true = np.asarray(y_true).reshape(-1)
    y_pred = np.asarray(y_pred).reshape(-1)
    mse = mean_squared_error(y_true, y_pred)
    rmse = math.sqrt(mse)
    denom = np.sum((y_true - np.mean(y_true)) ** 2) + 1e-8
    rrse = math.sqrt(float(np.sum((y_true - y_pred) ** 2) / denom))
    if np.std(y_true) < 1e-8 or np.std(y_pred) < 1e-8:
        corr = 0.0
    else:
        corr = float(np.corrcoef(y_true, y_pred)[0, 1])
    return {
        "MAE": float(mean_absolute_error(y_true, y_pred)),
        "RMSE": float(rmse),
        "RRSE": float(rrse),
        "CORR": float(corr),
        "R2": float(r2_score(y_true, y_pred)),
        "sMAPE": float(np.mean(2.0 * np.abs(y_true - y_pred) / (np.abs(y_true) + np.abs(y_pred) + 1e-8)) * 100),
        "WAPE": float(np.sum(np.abs(y_true - y_pred)) / (np.sum(np.abs(y_true)) + 1e-8) * 100),
    }


def seasonal_naive_forecast(history: np.ndarray, horizon: int, period: int) -> np.ndarray:
    history = np.asarray(history, dtype=np.float32)
    if len(history) == 0:
        return np.zeros(horizon, dtype=np.float32)
    if len(history) < period:
        return np.repeat(history[-1], horizon).astype(np.float32)
    last_period = history[-period:]
    reps = int(np.ceil(horizon / period))
    return np.tile(last_period, reps)[:horizon].astype(np.float32)


def maybe_sample_indices(n: int, max_n: Optional[int], seed: int = 42) -> np.ndarray:
    idx = np.arange(n)
    if max_n is not None and n > max_n:
        rng = np.random.default_rng(seed)
        idx = np.sort(rng.choice(idx, size=max_n, replace=False))
    return idx


import glob
import gzip
import shutil
import zipfile


# ============================================================
# 2b. Robust data discovery
# ============================================================
# The previous run printed "CSV files visible under /kaggle/input: 0", which
# means no dataset was mounted at execution time, so every dataset was skipped.
# This section:
#   (a) searches several roots, not only /kaggle/input,
#   (b) accepts .csv/.tsv/.txt/.xlsx/.parquet and auto-extracts .zip/.gz,
#   (c) matches files by filename and by column schema, at any depth,
#   (d) prints a full inventory of what it actually saw,
#   (e) optionally downloads the public datasets with kagglehub, and
#   (f) stops the run instead of silently producing an empty experiment.

DATA_SEARCH_ROOTS: List[str] = [
    "/kaggle/input",
    "/kaggle/working",
    os.getcwd(),
]
EXTRA_DATA_DIRS: List[str] = []          # add your own folders here if needed
EXTRACT_DIR = os.path.join(OUT_DIR, "extracted_inputs")
STRICT_DATA_CHECK = True                 # stop the run when no dataset resolves
TRY_KAGGLEHUB_FALLBACK = True            # requires Internet = ON in the notebook settings

TABULAR_EXTS = (".csv", ".tsv", ".txt", ".parquet", ".xlsx", ".xls")
ARCHIVE_EXTS = (".zip", ".gz")
SKIP_DIR_NAMES = {
    "__pycache__", "site-packages", "dist-packages", "node_modules",
    ".git", ".ipynb_checkpoints", "venv", ".venv", "env",
}

# Best-effort public sources used only when a file cannot be found locally.
# Edit these slugs to match the datasets you attached if a download fails.
KAGGLEHUB_FALLBACKS: Dict[str, List[str]] = {
    "DailyClimate": ["sumanthvrao/daily-climate-time-series-data"],
    "PJMW_Hourly": ["robikscube/hourly-energy-consumption"],
    "Traffic": [
        "anshtanwar/metro-interstate-traffic-volume",
        "ramyahr/metro-interstate-traffic-volume",
    ],
    "Demand": [
        "shelvigarg/store-item-demand-forecasting-challenge",
        "prathameshgadekar/store-item-demand-forecasting-challenge",
    ],
}


def _is_excluded_path(path: str) -> bool:
    """Ignore library folders and this run's own outputs, so only real inputs match."""
    abs_path = os.path.abspath(path)
    parts = set(abs_path.split(os.sep))
    if parts & SKIP_DIR_NAMES or any(p.endswith((".dist-info", ".egg-info")) for p in parts):
        return True
    abs_out = os.path.abspath(OUT_DIR)
    abs_extract = os.path.abspath(EXTRACT_DIR)
    if abs_path.startswith(abs_extract + os.sep):
        return False
    return abs_path.startswith(abs_out + os.sep)


def _dir_is_scannable(dirname: str) -> bool:
    return not dirname.startswith(".") and dirname not in SKIP_DIR_NAMES


def data_roots() -> List[str]:
    """Search roots, always including EXTRACT_DIR so unzipped files stay visible."""
    roots, seen = [], set()
    for root in list(DATA_SEARCH_ROOTS) + list(EXTRA_DATA_DIRS) + [EXTRACT_DIR]:
        if not root:
            continue
        abs_root = os.path.abspath(root)
        if abs_root in seen or not os.path.isdir(abs_root):
            continue
        seen.add(abs_root)
        roots.append(abs_root)
    return roots


def _walk_files(root: str, exts: Tuple[str, ...]) -> List[str]:
    found = []
    if not root or not os.path.isdir(root):
        return found
    for dirpath, dirnames, filenames in os.walk(root):
        dirnames[:] = [d for d in dirnames if _dir_is_scannable(d)]
        for filename in filenames:
            if not filename.lower().endswith(exts):
                continue
            path = os.path.join(dirpath, filename)
            if _is_excluded_path(path):
                continue
            found.append(path)
    return sorted(set(found))


def list_input_csvs(root: str = "/kaggle/input") -> List[str]:
    """Kept for backward compatibility: tabular files under one root."""
    return _walk_files(root, TABULAR_EXTS)


def list_all_tabular_files() -> List[str]:
    files: List[str] = []
    for root in data_roots():
        files.extend(_walk_files(root, TABULAR_EXTS))
    return sorted(set(files))


def print_input_inventory(max_files: int = 120) -> None:
    """Print what is actually visible. This is the diagnostic to read when a dataset is skipped."""
    for root in ["/kaggle/input"] + [r for r in data_roots() if r != "/kaggle/input"]:
        if not os.path.isdir(root):
            print(f"[inventory] {root}: does not exist")
            continue
        # For /kaggle/input list every file whatever the extension; elsewhere list
        # only data-like files, otherwise the listing is dominated by library code.
        list_everything = os.path.abspath(root) == "/kaggle/input"
        all_files = []
        for dirpath, dirnames, filenames in os.walk(root):
            dirnames[:] = [d for d in dirnames if _dir_is_scannable(d)]
            for filename in filenames:
                path = os.path.join(dirpath, filename)
                if _is_excluded_path(path):
                    continue
                if list_everything or filename.lower().endswith(TABULAR_EXTS + ARCHIVE_EXTS):
                    all_files.append(path)
        tabular = [p for p in all_files if p.lower().endswith(TABULAR_EXTS)]
        kind = "files" if list_everything else "data-like files"
        print(f"[inventory] {root}: {len(all_files)} {kind}, {len(tabular)} tabular")
        cap = max_files if list_everything else min(max_files, 40)
        for path in sorted(all_files)[:cap]:
            try:
                size_mb = os.path.getsize(path) / 1e6
                print(f"    - {path}  ({size_mb:.2f} MB)")
            except OSError:
                print(f"    - {path}")
        if len(all_files) > cap:
            print(f"    ... {len(all_files) - cap} more not printed")


def print_input_csvs(root: str = "/kaggle/input", max_files: int = 80) -> None:
    csv_paths = list_input_csvs(root)
    print(f"Tabular files visible under {root}: {len(csv_paths)}")
    for path in csv_paths[:max_files]:
        print(f"  - {path}")
    if len(csv_paths) > max_files:
        print(f"  ... {len(csv_paths) - max_files} more files not printed")


def extract_archives(verbose: bool = True) -> List[str]:
    """Extract data archives (.zip / .csv.gz) found in the input folders into EXTRACT_DIR."""
    archive_roots = [r for r in data_roots() if r == "/kaggle/input" or r in [os.path.abspath(p) for p in EXTRA_DATA_DIRS]]
    if not archive_roots:
        archive_roots = data_roots()
    archives: List[str] = []
    for root in archive_roots:
        archives.extend(_walk_files(root, ARCHIVE_EXTS))
    # Only archives that plausibly hold a table: foo.zip, foo.csv.gz, foo.txt.gz ...
    archives = [
        a for a in archives
        if a.lower().endswith(".zip")
        or (a.lower().endswith(".gz") and a.lower()[:-3].endswith(TABULAR_EXTS))
    ]
    if not archives:
        return []

    os.makedirs(EXTRACT_DIR, exist_ok=True)
    extracted: List[str] = []
    for archive in sorted(set(archives)):
        try:
            if archive.lower().endswith(".zip"):
                target = os.path.join(EXTRACT_DIR, os.path.splitext(os.path.basename(archive))[0])
                if os.path.isdir(target) and os.listdir(target):
                    extracted.append(target)
                    continue
                os.makedirs(target, exist_ok=True)
                with zipfile.ZipFile(archive) as zf:
                    zf.extractall(target)
                extracted.append(target)
            else:  # plain .gz
                target = os.path.join(EXTRACT_DIR, os.path.basename(archive)[:-3])
                if not os.path.exists(target):
                    with gzip.open(archive, "rb") as fin, open(target, "wb") as fout:
                        shutil.copyfileobj(fin, fout)
                extracted.append(target)
            if verbose:
                print(f"Extracted archive: {archive} -> {extracted[-1]}")
        except Exception as exc:
            print(f"Could not extract {archive}: {repr(exc)}")
    return extracted


def read_table(path: str, nrows: Optional[int] = None) -> pd.DataFrame:
    """Read csv / tsv / txt / xlsx / parquet with one call."""
    low = path.lower()
    if low.endswith(".parquet"):
        df = pd.read_parquet(path)
        return df.head(nrows) if nrows else df
    if low.endswith((".xlsx", ".xls")):
        return pd.read_excel(path, nrows=nrows)
    if low.endswith(".tsv"):
        return pd.read_csv(path, sep="\t", nrows=nrows)
    if low.endswith(".txt"):
        return pd.read_csv(path, sep=None, engine="python", nrows=nrows)
    return pd.read_csv(path, nrows=nrows)


def robust_read_csv(path: str) -> pd.DataFrame:
    if not os.path.exists(path):
        raise FileNotFoundError(path)
    return read_table(path)


def resolve_case_insensitive_path(path: str) -> Optional[str]:
    if not path:
        return None
    if os.path.exists(path):
        return path
    norm = os.path.normpath(path)
    if not norm.startswith(os.sep):
        return None

    current = os.sep
    for part in [p for p in norm.split(os.sep) if p]:
        try:
            entries = os.listdir(current)
        except Exception:
            return None
        match = None
        if part in entries:
            match = part
        else:
            part_lower = part.lower()
            for entry in entries:
                if entry.lower() == part_lower:
                    match = entry
                    break
        if match is None:
            return None
        current = os.path.join(current, match)
    return current if os.path.exists(current) else None


def _normalise_columns(cols) -> set:
    return {str(c).strip().lower() for c in cols}


def csv_has_columns(path: str, required_columns: List[str]) -> bool:
    try:
        header = read_table(path, nrows=5)
    except Exception:
        return False
    available = _normalise_columns(header.columns)
    required = _normalise_columns(required_columns)
    return required.issubset(available)


def _path_match_score(path: str, preferred_keywords: List[str]) -> int:
    lower_path = path.lower()
    score = sum(10 for key in preferred_keywords if key.lower() in lower_path)
    base = os.path.basename(path).lower()
    score += sum(3 for key in preferred_keywords if key.lower() in base)
    return score


def find_csv_file(
    root: str,
    candidate_filenames: List[str],
    required_columns: List[str],
    preferred_keywords: List[str],
    allow_schema_fallback: bool = True,
) -> Optional[str]:
    """Search every configured root (the `root` argument is kept for compatibility)."""
    csv_paths = list_all_tabular_files()
    if root and os.path.isdir(root):
        csv_paths = sorted(set(csv_paths) | set(_walk_files(os.path.abspath(root), TABULAR_EXTS)))
    if not csv_paths:
        return None

    candidate_stems = {os.path.splitext(name)[0].lower() for name in candidate_filenames}
    exact = [p for p in csv_paths if os.path.splitext(os.path.basename(p))[0].lower() in candidate_stems]
    exact = [p for p in exact if csv_has_columns(p, required_columns)]
    if exact:
        exact.sort(key=lambda p: _path_match_score(p, preferred_keywords), reverse=True)
        return exact[0]

    if not allow_schema_fallback:
        return None

    schema_matches = [p for p in csv_paths if csv_has_columns(p, required_columns)]
    if not schema_matches:
        return None
    schema_matches.sort(key=lambda p: _path_match_score(p, preferred_keywords), reverse=True)
    return schema_matches[0]


DATASET_FILE_SPECS: Dict[str, Dict[str, Dict]] = {
    "DailyClimate": {
        "train_path": {
            "path_candidates": [
                "/kaggle/input/dailyclimate/DailyDelhiClimateTrain.csv",
                "/kaggle/input/DailyClimate/DailyDelhiClimateTrain.csv",
                "/kaggle/input/daily-climate-time-series-data/DailyDelhiClimateTrain.csv",
            ],
            "candidate_filenames": ["DailyDelhiClimateTrain.csv"],
            "required_columns": ["date", "meantemp"],
            "preferred_keywords": ["daily", "climate", "delhi", "train"],
        },
        "test_path": {
            "path_candidates": [
                "/kaggle/input/dailyclimate/DailyDelhiClimateTest.csv",
                "/kaggle/input/DailyClimate/DailyDelhiClimateTest.csv",
                "/kaggle/input/daily-climate-time-series-data/DailyDelhiClimateTest.csv",
            ],
            "candidate_filenames": ["DailyDelhiClimateTest.csv"],
            "required_columns": ["date", "meantemp"],
            "preferred_keywords": ["daily", "climate", "delhi", "test"],
        },
    },
    "Demand": {
        "train_path": {
            "path_candidates": [
                "/kaggle/input/demanddataset/train.csv",
                "/kaggle/input/Demanddataset/train.csv",
            ],
            "candidate_filenames": ["train.csv"],
            "required_columns": ["date", "store", "item", "sales"],
            "preferred_keywords": ["demand", "forecasting", "sales", "store", "item", "train"],
        },
        "test_path": {
            "path_candidates": [
                "/kaggle/input/demanddataset/test.csv",
                "/kaggle/input/Demanddataset/test.csv",
            ],
            "candidate_filenames": ["test.csv"],
            "required_columns": ["date", "store", "item"],
            "preferred_keywords": ["demand", "forecasting", "sales", "store", "item", "test"],
            "optional": True,
            "allow_schema_fallback": False,
        },
    },
    "PJMW_Hourly": {
        "single_path": {
            "path_candidates": [
                "/kaggle/input/timeseriesdataset/PJMW_hourly.csv",
                "/kaggle/input/Timeseriesdataset/PJMW_hourly.csv",
                "/kaggle/input/hourly-energy-consumption/PJMW_hourly.csv",
            ],
            "candidate_filenames": ["PJMW_hourly.csv"],
            "required_columns": ["Datetime", "PJMW_MW"],
            "preferred_keywords": ["pjmw", "hourly", "energy"],
        },
    },
    "Traffic": {
        "single_path": {
            "path_candidates": [
                "/kaggle/input/timeseriesdataset/Traffic.csv",
                "/kaggle/input/Timeseriesdataset/Traffic.csv",
                "/kaggle/input/timeseriesdataset/Metro_Interstate_Traffic_Volume.csv",
                "/kaggle/input/Timeseriesdataset/Metro_Interstate_Traffic_Volume.csv",
            ],
            "candidate_filenames": ["Traffic.csv", "Metro_Interstate_Traffic_Volume.csv"],
            "required_columns": ["date_time", "traffic_volume"],
            "preferred_keywords": ["traffic", "metro", "interstate"],
        },
    },
}


def _resolve_pass(only_names: Optional[List[str]] = None, verbose: bool = True) -> List[str]:
    """Resolve dataset file paths in place. Returns names still unresolved."""
    unresolved: List[str] = []
    for dcfg in DATASETS:
        name = dcfg["name"]
        if only_names is not None and name not in only_names:
            continue
        ok = True
        for key, spec in DATASET_FILE_SPECS.get(name, {}).items():
            found = None
            for candidate in [dcfg.get(key)] + spec.get("path_candidates", []):
                resolved = resolve_case_insensitive_path(candidate)
                if resolved:
                    found = resolved
                    break
            if found is None:
                found = find_csv_file(
                    root="/kaggle/input",
                    candidate_filenames=spec["candidate_filenames"],
                    required_columns=spec["required_columns"],
                    preferred_keywords=spec["preferred_keywords"],
                    allow_schema_fallback=spec.get("allow_schema_fallback", True),
                )
            if found:
                old = dcfg.get(key)
                dcfg[key] = found
                if verbose and old != found:
                    print(f"Resolved {name} {key}: {found}")
            elif not spec.get("optional", False):
                ok = False
                if verbose:
                    print(f"Could not resolve {name} {key} (current path: {dcfg.get(key)})")
        if not ok:
            unresolved.append(name)
    return unresolved


def download_with_kagglehub(dataset_names: List[str]) -> bool:
    """Best-effort download of the public datasets. Needs Internet = ON."""
    try:
        import kagglehub
    except Exception:
        print("kagglehub is not available, skipping the download fallback.")
        return False

    downloaded_any = False
    for name in dataset_names:
        for slug in KAGGLEHUB_FALLBACKS.get(name, []):
            try:
                print(f"Downloading {name} from kagglehub: {slug}")
                path = kagglehub.dataset_download(slug)
                if path and os.path.isdir(path):
                    if path not in EXTRA_DATA_DIRS:
                        EXTRA_DATA_DIRS.append(path)
                    print(f"  downloaded to: {path}")
                    downloaded_any = True
                    break
            except Exception as exc:
                print(f"  failed ({slug}): {repr(exc)}")
    return downloaded_any


def auto_resolve_kaggle_input_paths(root: str = "/kaggle/input") -> None:
    print("=" * 100)
    print("Data discovery")
    print("=" * 100)
    print_input_inventory()

    extracted = extract_archives()
    if extracted:
        print(f"Extracted {len(extracted)} archive(s) into {EXTRACT_DIR}")

    unresolved = _resolve_pass()

    if unresolved and TRY_KAGGLEHUB_FALLBACK:
        print(f"\nDatasets still unresolved: {unresolved}")
        print("Trying the kagglehub download fallback (requires Internet = ON).")
        if download_with_kagglehub(unresolved):
            extract_archives()
            unresolved = _resolve_pass(only_names=unresolved)

    resolved = [d["name"] for d in DATASETS if d["name"] not in unresolved]
    print(f"\nResolved datasets:   {resolved if resolved else 'none'}")
    print(f"Unresolved datasets: {unresolved if unresolved else 'none'}")


DATA_HELP = """
No dataset file could be located, so the experiment would produce nothing.

Fix it in the Kaggle notebook editor:
  1. Right sidebar -> Input -> "+ Add Input" -> attach each dataset.
  2. After attaching, restart the session:
     Run -> "Restart & clear cell outputs"   (a running session does not
     always see inputs mounted after it started - this is the usual cause
     of "0 files under /kaggle/input").
  3. Re-run this cell and read the [inventory] lines printed above: they
     list every file the notebook can actually see.
  4. If the folder names differ, copy the real paths from the inventory
     into DATASETS, or add their parent folder to EXTRA_DATA_DIRS.

Alternative without attaching anything:
  Settings -> Internet = ON, keep TRY_KAGGLEHUB_FALLBACK = True, and the
  notebook will download the public datasets itself.
"""


def assert_datasets_available() -> List[Dict]:
    """Return the runnable datasets; raise with instructions when there are none."""
    runnable = []
    for dcfg in DATASETS:
        missing = [p for p in dataset_paths(dcfg) if not p or not os.path.exists(p)]
        if not missing:
            runnable.append(dcfg)
    if not runnable:
        print(DATA_HELP)
        if STRICT_DATA_CHECK:
            raise RuntimeError("No dataset files found. See the instructions printed above.")
    elif len(runnable) < len(DATASETS):
        names = [d["name"] for d in DATASETS if d not in runnable]
        print(f"\nWARNING: running with a partial dataset set. Missing: {names}")
    return runnable


# ============================================================
# 3. Data loading
# ============================================================

def read_and_clean_single(path: str, time_col: str, target_col: str, freq: str) -> pd.DataFrame:
    df = robust_read_csv(path)
    df[time_col] = pd.to_datetime(df[time_col], errors="coerce")
    df[target_col] = pd.to_numeric(df[target_col], errors="coerce")
    df = df.dropna(subset=[time_col, target_col]).sort_values(time_col)
    df = df[[time_col, target_col]].drop_duplicates(subset=[time_col], keep="last")
    df = df.set_index(time_col).asfreq(freq)
    df[target_col] = df[target_col].ffill()
    df = df.dropna(subset=[target_col])
    return df.reset_index()


def load_single_dataset(dcfg: Dict) -> Tuple[pd.DataFrame, int, Optional[int]]:
    tcol, ycol, freq = dcfg["time_col"], dcfg["target_col"], dcfg["freq"]
    if "train_path" in dcfg and "test_path" in dcfg:
        train_df = read_and_clean_single(dcfg["train_path"], tcol, ycol, freq)
        test_df = read_and_clean_single(dcfg["test_path"], tcol, ycol, freq)
        df = pd.concat([train_df, test_df], axis=0, ignore_index=True)
        df = df.drop_duplicates(subset=[tcol], keep="last").sort_values(tcol).reset_index(drop=True)
        train_last_time = train_df[tcol].max()
        test_first_time = test_df[tcol].min()
        train_end_index = int(np.searchsorted(df[tcol].values, np.datetime64(train_last_time), side="right"))
        test_start_index = int(np.searchsorted(df[tcol].values, np.datetime64(test_first_time), side="left"))
        return df, train_end_index, test_start_index
    df = read_and_clean_single(dcfg["single_path"], tcol, ycol, freq)
    return df, -1, None


def read_panel_train_test(dcfg: Dict) -> Tuple[pd.DataFrame, int, Optional[int], Dict[Tuple, int]]:
    """Panel loader for Demand dataset. Uses only rows with known target values."""
    tcol, ycol, id_cols, freq = dcfg["time_col"], dcfg["target_col"], dcfg["id_cols"], dcfg["freq"]
    train = robust_read_csv(dcfg["train_path"])
    train[tcol] = pd.to_datetime(train[tcol], errors="coerce")
    train[ycol] = pd.to_numeric(train[ycol], errors="coerce")
    train = train.dropna(subset=[tcol, ycol] + id_cols).sort_values(tcol)

    # Keep a subset of series for faster initial development; set max_series=None for full run.
    series_keys = train[id_cols].drop_duplicates().apply(lambda r: tuple(r.values.tolist()), axis=1).tolist()
    max_series = dcfg["config"].max_series
    if max_series is not None and len(series_keys) > max_series:
        series_keys = series_keys[:max_series]
        key_set = set(series_keys)
        train = train[train[id_cols].apply(lambda r: tuple(r.values.tolist()) in key_set, axis=1)].copy()

    series_to_id = {k: i for i, k in enumerate(series_keys)}
    records = []
    for key, g in train.groupby(id_cols, sort=False):
        if not isinstance(key, tuple):
            key = (key,)
        if key not in series_to_id:
            continue
        s = g[[tcol, ycol]].copy()
        s = s.drop_duplicates(subset=[tcol], keep="last").set_index(tcol).asfreq(freq)
        s[ycol] = s[ycol].ffill()
        s = s.dropna(subset=[ycol])
        s = s.reset_index()
        s["series_id"] = series_to_id[key]
        records.append(s)
    full = pd.concat(records, ignore_index=True)
    return full, -1, None, series_to_id


def chronological_split_indices(
    n: int,
    lookback: int,
    horizon: int,
    train_end_index: int = -1,
    test_start_index: Optional[int] = None,
    train_ratio: float = 0.70,
    val_ratio: float = 0.15,
) -> Tuple[int, int]:
    if test_start_index is not None and train_end_index > 0:
        test_start = test_start_index
        val_len = max(int(train_end_index * val_ratio), 3 * horizon)
        val_start = max(lookback + 1, train_end_index - val_len)
        return val_start, test_start
    val_start = int(n * train_ratio)
    test_start = int(n * (train_ratio + val_ratio))
    val_start = max(val_start, lookback + horizon + 1)
    test_start = max(test_start, val_start + horizon + 1)
    return val_start, test_start


# ============================================================
# 4. Windowing and wavelet decomposition
# ============================================================

def build_windows_single(y: np.ndarray, cfg: ModelConfig, stride: int = 1) -> Tuple[np.ndarray, np.ndarray, np.ndarray, np.ndarray]:
    L, H = cfg.lookback, cfg.horizon
    X, Y, T, SID = [], [], [], []
    for start in range(0, len(y) - L - H + 1, stride):
        target_start = start + L
        X.append(y[start:start + L])
        Y.append(y[target_start:target_start + H])
        T.append(target_start)
        SID.append(0)
    return np.asarray(X, np.float32), np.asarray(Y, np.float32), np.asarray(T, np.int64), np.asarray(SID, np.int64)


def build_windows_panel(df: pd.DataFrame, y_col: str, cfg: ModelConfig) -> Tuple[np.ndarray, np.ndarray, np.ndarray, np.ndarray, Dict[int, int]]:
    """Build windows per series. T is local target start; split is within each series."""
    X_all, Y_all, T_all, SID_all = [], [], [], []
    series_lengths = {}
    for sid, g in df.groupby("series_id", sort=False):
        y = g[y_col].values.astype(np.float32)
        series_lengths[int(sid)] = len(y)
        X, Y, T, SID = build_windows_single(y, cfg, stride=cfg.stride)
        if len(Y) == 0:
            continue
        SID[:] = int(sid)
        X_all.append(X); Y_all.append(Y); T_all.append(T); SID_all.append(SID)
    if not X_all:
        raise RuntimeError("No panel windows were created. Reduce lookback/horizon or check the dataset.")
    return (
        np.concatenate(X_all, axis=0),
        np.concatenate(Y_all, axis=0),
        np.concatenate(T_all, axis=0),
        np.concatenate(SID_all, axis=0),
        series_lengths,
    )


def swt_decompose_window(x: np.ndarray, max_levels: int, wavelet: str) -> np.ndarray:
    x = np.asarray(x, dtype=np.float32)
    if not HAS_PYWT:
        return fallback_multiscale_window(x, max_levels)
    n = len(x)
    possible = pywt.swt_max_level(n)
    level = int(min(max_levels, max(1, possible)))
    block = 2 ** level
    need = int(np.ceil(n / block) * block)
    x_pad = np.pad(x, (0, max(0, need - n)), mode="edge") if need > n else x
    coeffs = pywt.swt(x_pad, wavelet=wavelet, level=level, start_level=0, trim_approx=False)
    approx = coeffs[-1][0][:n].astype(np.float32)
    details = [cD[:n].astype(np.float32) for (_, cD) in coeffs]
    return np.vstack([approx[None, :]] + [d[None, :] for d in details]).astype(np.float32)


def fallback_multiscale_window(x: np.ndarray, max_levels: int) -> np.ndarray:
    x = np.asarray(x, dtype=np.float32)
    levels = int(max(1, max_levels))
    approx = x.copy()
    details = []
    for level in range(1, levels + 1):
        window = min(len(x), max(3, 2 ** (level + 1) - 1))
        if window % 2 == 0:
            window = max(3, window - 1)
        pad = window // 2
        padded = np.pad(approx, (pad, pad), mode="edge")
        kernel = np.ones(window, dtype=np.float32) / float(window)
        smooth = np.convolve(padded, kernel, mode="valid").astype(np.float32)
        details.append((approx - smooth).astype(np.float32))
        approx = smooth
    return np.vstack([approx[None, :]] + [d[None, :] for d in details]).astype(np.float32)


def build_wavelet_tensor(X_scaled: np.ndarray, cfg: ModelConfig) -> np.ndarray:
    comps = [swt_decompose_window(x, cfg.wavelet_levels, cfg.wavelet) for x in X_scaled]
    return np.asarray(comps, dtype=np.float32)


# ============================================================
# 5. Torch dataset
# ============================================================

class ForecastDataset(Dataset):
    def __init__(self, x_raw: np.ndarray, x_wav: np.ndarray, y: np.ndarray, series_id: np.ndarray):
        self.x_raw = torch.tensor(x_raw, dtype=torch.float32)
        self.x_wav = torch.tensor(x_wav, dtype=torch.float32)
        self.y = torch.tensor(y, dtype=torch.float32)
        self.series_id = torch.tensor(series_id, dtype=torch.long)

    def __len__(self):
        return len(self.y)

    def __getitem__(self, idx):
        return self.x_raw[idx], self.x_wav[idx], self.y[idx], self.series_id[idx]


# ============================================================
# 6. Model blocks
# ============================================================

class PositionalEncoding:
    @staticmethod
    def build(n_tokens: int, d_model: int, device: torch.device) -> torch.Tensor:
        pos = torch.arange(0, n_tokens, dtype=torch.float32, device=device).unsqueeze(1)
        div = torch.exp(torch.arange(0, d_model, 2, device=device).float() * (-math.log(10000.0) / d_model))
        pe = torch.zeros(n_tokens, d_model, device=device)
        pe[:, 0::2] = torch.sin(pos * div)
        pe[:, 1::2] = torch.cos(pos * div)
        return pe


class PatchEncoder(nn.Module):
    def __init__(self, patch_len: int, d_model: int, n_heads: int, enc_layers: int, ff_dim: int, dropout: float):
        super().__init__()
        self.patch_len = patch_len
        self.proj = nn.Linear(patch_len, d_model)
        layer = nn.TransformerEncoderLayer(
            d_model=d_model,
            nhead=n_heads,
            dim_feedforward=ff_dim,
            dropout=dropout,
            batch_first=True,
            activation="gelu",
            norm_first=True,
        )
        self.encoder = nn.TransformerEncoder(layer, num_layers=enc_layers)
        self.norm = nn.LayerNorm(d_model)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        B, L = x.shape
        n_patches = L // self.patch_len
        x = x[:, : n_patches * self.patch_len]
        x = x.reshape(B, n_patches, self.patch_len)
        z = self.proj(x)
        z = z + PositionalEncoding.build(n_patches, z.size(-1), z.device).unsqueeze(0)
        z = self.encoder(z)
        return self.norm(z)


class SeriesEmbeddingMixin(nn.Module):
    def _init_series_embedding(self, n_series: int, d_model: int):
        self.n_series = n_series
        self.series_emb = nn.Embedding(n_series, d_model) if n_series > 1 else None

    def _add_series_embedding(self, feat: torch.Tensor, series_id: Optional[torch.Tensor]):
        if self.series_emb is not None and series_id is not None:
            feat = feat + self.series_emb(series_id)
        return feat



class RevIN1D(nn.Module):
    """
    Lightweight RevIN-style normalization for univariate forecasting.
    The model sees instance-normalized input. Its output is denormalized back
    to the same globally scaled space before loss calculation.
    """
    def __init__(self, eps: float = 1e-5, affine: bool = True):
        super().__init__()
        self.eps = eps
        self.affine = affine
        if affine:
            self.gamma = nn.Parameter(torch.ones(1))
            self.beta = nn.Parameter(torch.zeros(1))

    def normalize_raw(self, x: torch.Tensor):
        mean = x.mean(dim=1, keepdim=True).detach()
        std = x.std(dim=1, keepdim=True).detach().clamp_min(self.eps)
        z = (x - mean) / std
        if self.affine:
            z = z * self.gamma + self.beta
        return z, mean, std

    def normalize_wavelet(self, x_wav: torch.Tensor, mean: torch.Tensor, std: torch.Tensor):
        # Approximation component is on the same scale as the raw signal.
        # Detail components are centered around zero; scaling by instance std is sufficient.
        z = x_wav.clone()
        z[:, 0, :] = (z[:, 0, :] - mean) / std
        if z.size(1) > 1:
            z[:, 1:, :] = z[:, 1:, :] / std.unsqueeze(1)
        if self.affine:
            z = z * self.gamma + self.beta
        return z

    def denormalize_pred(self, y: torch.Tensor, mean: torch.Tensor, std: torch.Tensor):
        if self.affine:
            y = (y - self.beta) / (self.gamma + self.eps)
        return y * std + mean

class RawPatchExpert(SeriesEmbeddingMixin):
    def __init__(self, cfg: ModelConfig, n_series: int = 1):
        super().__init__()
        self.encoder = PatchEncoder(cfg.patch_len, cfg.d_model, cfg.n_heads, cfg.enc_layers, cfg.ff_dim, cfg.dropout)
        self._init_series_embedding(n_series, cfg.d_model)
        self.head = nn.Sequential(
            nn.LayerNorm(cfg.d_model),
            nn.Linear(cfg.d_model, cfg.d_model // 2),
            nn.GELU(),
            nn.Dropout(cfg.dropout),
            nn.Linear(cfg.d_model // 2, cfg.horizon),
        )

    def forward(self, x_raw: torch.Tensor, series_id: Optional[torch.Tensor] = None) -> Tuple[torch.Tensor, torch.Tensor]:
        tokens = self.encoder(x_raw)
        feat = tokens.mean(dim=1)
        feat = self._add_series_embedding(feat, series_id)
        pred = self.head(feat)
        return pred, feat


class CrossScaleAttentionBlock(nn.Module):
    def __init__(self, d_model: int, n_heads: int, ff_dim: int, dropout: float):
        super().__init__()
        self.attn = nn.MultiheadAttention(d_model, n_heads, dropout=dropout, batch_first=True)
        self.norm1 = nn.LayerNorm(d_model)
        self.ff = nn.Sequential(
            nn.Linear(d_model, ff_dim), nn.GELU(), nn.Dropout(dropout), nn.Linear(ff_dim, d_model)
        )
        self.norm2 = nn.LayerNorm(d_model)
        self.drop = nn.Dropout(dropout)

    def forward(self, query_tokens: torch.Tensor, context_tokens: torch.Tensor) -> torch.Tensor:
        attn_out, _ = self.attn(query_tokens, context_tokens, context_tokens)
        x = self.norm1(query_tokens + self.drop(attn_out))
        y = self.ff(x)
        return self.norm2(x + self.drop(y))


class WaveletCrossScaleExpert(SeriesEmbeddingMixin):
    def __init__(self, n_scales: int, cfg: ModelConfig, n_series: int = 1):
        super().__init__()
        self.n_scales = n_scales
        self.branches = nn.ModuleList([
            PatchEncoder(cfg.patch_len, cfg.d_model, cfg.n_heads, cfg.enc_layers, cfg.ff_dim, cfg.dropout)
            for _ in range(n_scales)
        ])
        self.cross = CrossScaleAttentionBlock(cfg.d_model, cfg.n_heads, cfg.ff_dim, cfg.dropout)
        fusion_layer = nn.TransformerEncoderLayer(
            d_model=cfg.d_model, nhead=cfg.n_heads, dim_feedforward=cfg.ff_dim,
            dropout=cfg.dropout, batch_first=True, activation="gelu", norm_first=True,
        )
        self.fusion = nn.TransformerEncoder(fusion_layer, num_layers=1)
        self._init_series_embedding(n_series, cfg.d_model)
        self.head = nn.Sequential(
            nn.LayerNorm(cfg.d_model),
            nn.Linear(cfg.d_model, cfg.d_model // 2),
            nn.GELU(),
            nn.Dropout(cfg.dropout),
            nn.Linear(cfg.d_model // 2, cfg.horizon),
        )

    def forward(self, x_wav: torch.Tensor, series_id: Optional[torch.Tensor] = None) -> Tuple[torch.Tensor, torch.Tensor]:
        tokens = [self.branches[s](x_wav[:, s, :]) for s in range(self.n_scales)]
        approx_tokens = tokens[0]
        if self.n_scales > 1:
            detail_tokens = torch.cat(tokens[1:], dim=1)
            approx_tokens = self.cross(approx_tokens, detail_tokens)
        fused = self.fusion(approx_tokens)
        feat = fused.mean(dim=1)
        feat = self._add_series_embedding(feat, series_id)
        pred = self.head(feat)
        return pred, feat


class TrendSeasonalExpert(SeriesEmbeddingMixin):
    def __init__(self, cfg: ModelConfig, n_series: int = 1):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(14, cfg.d_model),
            nn.GELU(),
            nn.Dropout(cfg.dropout),
            nn.Linear(cfg.d_model, cfg.d_model // 2),
            nn.GELU(),
            nn.Linear(cfg.d_model // 2, cfg.horizon),
        )
        self.feat_proj = nn.Linear(14, cfg.d_model)
        self._init_series_embedding(n_series, cfg.d_model)

    def forward(self, x_raw: torch.Tensor, series_id: Optional[torch.Tensor] = None) -> Tuple[torch.Tensor, torch.Tensor]:
        B, L = x_raw.shape
        last = x_raw[:, -1]
        mean = x_raw.mean(dim=1)
        std = x_raw.std(dim=1)
        minv = x_raw.min(dim=1).values
        maxv = x_raw.max(dim=1).values
        first = x_raw[:, 0]
        slope = (last - first) / max(L - 1, 1)
        q10 = torch.quantile(x_raw, 0.10, dim=1)
        q25 = torch.quantile(x_raw, 0.25, dim=1)
        q50 = torch.quantile(x_raw, 0.50, dim=1)
        q75 = torch.quantile(x_raw, 0.75, dim=1)
        q90 = torch.quantile(x_raw, 0.90, dim=1)
        recent_mean = x_raw[:, -min(24, L):].mean(dim=1)
        recent_delta = last - recent_mean
        stats = torch.stack([last, mean, std, minv, maxv, first, slope, q10, q25, q50, q75, q90, recent_mean, recent_delta], dim=1)
        pred = self.net(stats)
        feat = self.feat_proj(stats)
        feat = self._add_series_embedding(feat, series_id)
        return pred, feat


class DynamicGatingNetwork(nn.Module):
    def __init__(self, cfg: ModelConfig, n_experts: int = 3):
        super().__init__()
        self.base_temperature = float(cfg.gate_temperature)
        self.use_learnable_temperature = bool(cfg.use_learnable_temperature)
        self.sparse_top_k = int(cfg.sparse_top_k or 0)
        self.n_experts = n_experts
        if self.use_learnable_temperature:
            # Softplus(log_temperature) gives a positive learned temperature.
            init = math.log(max(self.base_temperature, 1e-3))
            self.log_temperature = nn.Parameter(torch.tensor(init, dtype=torch.float32))
        else:
            self.register_buffer("fixed_temperature", torch.tensor(self.base_temperature, dtype=torch.float32))
        self.net = nn.Sequential(
            nn.Linear(cfg.d_model * n_experts, cfg.d_model),
            nn.LayerNorm(cfg.d_model),
            nn.GELU(),
            nn.Dropout(cfg.dropout),
            nn.Linear(cfg.d_model, cfg.d_model // 2),
            nn.GELU(),
            nn.Linear(cfg.d_model // 2, n_experts),
        )

    def _temperature(self):
        if self.use_learnable_temperature:
            return F.softplus(self.log_temperature).clamp(0.30, 5.00)
        return self.fixed_temperature.clamp(0.30, 5.00)

    def forward(self, features: List[torch.Tensor]) -> torch.Tensor:
        z = torch.cat(features, dim=1)
        logits = self.net(z) / self._temperature()
        weights = torch.softmax(logits, dim=1)
        # Sparse top-k gating: retains only strongest experts per sample and renormalizes.
        # This usually improves interpretability and prevents weak experts from adding noise.
        if 0 < self.sparse_top_k < self.n_experts:
            top_vals, top_idx = torch.topk(weights, k=self.sparse_top_k, dim=1)
            sparse = torch.zeros_like(weights).scatter(1, top_idx, top_vals)
            weights = sparse / sparse.sum(dim=1, keepdim=True).clamp_min(1e-8)
        return weights


class E3TSF(nn.Module):
    def __init__(self, n_scales: int, cfg: ModelConfig, n_series: int = 1):
        super().__init__()
        self.cfg = cfg
        self.use_revin = bool(cfg.use_revin)
        self.revin = RevIN1D() if self.use_revin else None
        self.raw_expert = RawPatchExpert(cfg, n_series=n_series)
        self.wavelet_expert = WaveletCrossScaleExpert(n_scales, cfg, n_series=n_series)
        self.trend_expert = TrendSeasonalExpert(cfg, n_series=n_series)
        self.gate = DynamicGatingNetwork(cfg, n_experts=3)

    def forward(self, x_raw: torch.Tensor, x_wav: torch.Tensor, series_id: Optional[torch.Tensor] = None, return_gate: bool = False):
        if self.use_revin:
            x_raw_model, mean, std = self.revin.normalize_raw(x_raw)
            x_wav_model = self.revin.normalize_wavelet(x_wav, mean, std)
        else:
            x_raw_model, x_wav_model = x_raw, x_wav
            mean, std = None, None

        p1, f1 = self.raw_expert(x_raw_model, series_id)
        p2, f2 = self.wavelet_expert(x_wav_model, series_id)
        p3, f3 = self.trend_expert(x_raw_model, series_id)
        weights = self.gate([f1, f2, f3])
        stacked = torch.stack([p1, p2, p3], dim=1)
        pred_norm = torch.sum(stacked * weights.unsqueeze(-1), dim=1)
        pred = self.revin.denormalize_pred(pred_norm, mean, std) if self.use_revin else pred_norm
        if return_gate:
            return pred, weights, stacked
        return pred


class E3TSFNoGateAverage(nn.Module):
    def __init__(self, n_scales: int, cfg: ModelConfig, n_series: int = 1):
        super().__init__()
        self.cfg = cfg
        self.use_revin = bool(cfg.use_revin)
        self.revin = RevIN1D() if self.use_revin else None
        self.raw_expert = RawPatchExpert(cfg, n_series=n_series)
        self.wavelet_expert = WaveletCrossScaleExpert(n_scales, cfg, n_series=n_series)
        self.trend_expert = TrendSeasonalExpert(cfg, n_series=n_series)

    def forward(self, x_raw: torch.Tensor, x_wav: torch.Tensor, series_id: Optional[torch.Tensor] = None, return_experts: bool = False):
        if self.use_revin:
            x_raw_model, mean, std = self.revin.normalize_raw(x_raw)
            x_wav_model = self.revin.normalize_wavelet(x_wav, mean, std)
        else:
            x_raw_model, x_wav_model = x_raw, x_wav
            mean, std = None, None

        p1, _ = self.raw_expert(x_raw_model, series_id)
        p2, _ = self.wavelet_expert(x_wav_model, series_id)
        p3, _ = self.trend_expert(x_raw_model, series_id)
        stacked = torch.stack([p1, p2, p3], dim=1)
        pred_norm = stacked.mean(dim=1)
        pred = self.revin.denormalize_pred(pred_norm, mean, std) if self.use_revin else pred_norm
        if return_experts:
            return pred, stacked
        return pred


class PatchTSTBaseline(nn.Module):
    def __init__(self, cfg: ModelConfig, n_series: int = 1):
        super().__init__()
        self.expert = RawPatchExpert(cfg, n_series=n_series)

    def forward(self, x_raw: torch.Tensor, x_wav: Optional[torch.Tensor] = None, series_id: Optional[torch.Tensor] = None):
        pred, _ = self.expert(x_raw, series_id)
        return pred


class DLinearBaseline(nn.Module):
    def __init__(self, cfg: ModelConfig, n_series: int = 1):
        super().__init__()
        self.lookback = cfg.lookback
        self.horizon = cfg.horizon
        self.kernel_size = max(3, min(cfg.seasonal_period, cfg.lookback))
        if self.kernel_size % 2 == 0:
            self.kernel_size -= 1
        self.trend_linear = nn.Linear(cfg.lookback, cfg.horizon)
        self.seasonal_linear = nn.Linear(cfg.lookback, cfg.horizon)
        self.series_bias = nn.Embedding(n_series, cfg.horizon) if n_series > 1 else None

    def moving_average(self, x: torch.Tensor) -> torch.Tensor:
        pad = self.kernel_size // 2
        x_pad = F.pad(x.unsqueeze(1), (pad, pad), mode="replicate")
        return F.avg_pool1d(x_pad, kernel_size=self.kernel_size, stride=1).squeeze(1)

    def forward(self, x_raw: torch.Tensor, x_wav: Optional[torch.Tensor] = None, series_id: Optional[torch.Tensor] = None):
        trend = self.moving_average(x_raw)
        seasonal = x_raw - trend
        out = self.trend_linear(trend) + self.seasonal_linear(seasonal)
        if self.series_bias is not None and series_id is not None:
            out = out + self.series_bias(series_id)
        return out


# ============================================================
# 7. Training and prediction
# ============================================================

def model_forward(model, x_raw, x_wav, sid, return_gate=False):
    if isinstance(model, E3TSF):
        return model(x_raw, x_wav, sid, return_gate=return_gate)
    return model(x_raw, x_wav, sid)


def training_loss(pred: torch.Tensor, y: torch.Tensor) -> torch.Tensor:
    # Robust but still sensitive to larger errors.
    return 0.70 * F.huber_loss(pred, y, delta=1.0) + 0.30 * F.mse_loss(pred, y)


def expert_diversity_penalty(expert_preds: torch.Tensor) -> torch.Tensor:
    # Encourage the three experts to avoid identical horizon-wise predictions.
    e = expert_preds - expert_preds.mean(dim=-1, keepdim=True)
    e = F.normalize(e, dim=-1)
    sim12 = (e[:, 0, :] * e[:, 1, :]).sum(dim=-1).abs().mean()
    sim13 = (e[:, 0, :] * e[:, 2, :]).sum(dim=-1).abs().mean()
    sim23 = (e[:, 1, :] * e[:, 2, :]).sum(dim=-1).abs().mean()
    return (sim12 + sim13 + sim23) / 3.0


def train_model(model: nn.Module, train_loader: DataLoader, val_loader: DataLoader, cfg: ModelConfig) -> nn.Module:
    optimizer = torch.optim.AdamW(model.parameters(), lr=cfg.lr, weight_decay=cfg.weight_decay)
    total_steps = max(1, cfg.epochs * len(train_loader))
    warmup_steps = max(1, int(0.08 * total_steps))

    def lr_lambda(step):
        if step < warmup_steps:
            return float(step + 1) / float(warmup_steps)
        progress = float(step - warmup_steps) / float(max(1, total_steps - warmup_steps))
        cosine = 0.5 * (1.0 + math.cos(math.pi * progress))
        return max(cfg.min_lr / cfg.lr, cosine)

    scheduler = torch.optim.lr_scheduler.LambdaLR(optimizer, lr_lambda=lr_lambda)
    scaler = make_amp_scaler(enabled=(cfg.use_amp and DEVICE == "cuda"))

    best_state = None
    best_val = float("inf")
    patience = cfg.early_stop
    global_step = 0

    for epoch in range(1, cfg.epochs + 1):
        model.train()
        train_losses = []
        for x_raw, x_wav, y, sid in train_loader:
            x_raw = x_raw.to(DEVICE, non_blocking=True)
            x_wav = x_wav.to(DEVICE, non_blocking=True)
            y = y.to(DEVICE, non_blocking=True)
            sid = sid.to(DEVICE, non_blocking=True)
            optimizer.zero_grad(set_to_none=True)

            with amp_autocast(enabled=(cfg.use_amp and DEVICE == "cuda")):
                if isinstance(model, E3TSF):
                    pred, gate, expert_preds = model(x_raw, x_wav, sid, return_gate=True)
                    loss = training_loss(pred, y)
                    entropy = -(gate * torch.log(gate + 1e-8)).sum(dim=1).mean()
                    diversity_penalty = expert_diversity_penalty(expert_preds)
                    # Balance: avoid global collapse to one expert while still allowing sample-specific routing.
                    mean_gate = gate.mean(dim=0)
                    uniform = torch.full_like(mean_gate, 1.0 / mean_gate.numel())
                    balance_penalty = F.mse_loss(mean_gate, uniform)
                    loss = loss - cfg.gate_entropy_lambda * entropy
                    loss = loss + cfg.gate_diversity_lambda * diversity_penalty
                    loss = loss + cfg.gate_balance_lambda * balance_penalty
                elif isinstance(model, E3TSFNoGateAverage):
                    pred, expert_preds = model(x_raw, x_wav, sid, return_experts=True)
                    loss = training_loss(pred, y)
                    loss = loss + cfg.gate_diversity_lambda * expert_diversity_penalty(expert_preds)
                else:
                    pred = model_forward(model, x_raw, x_wav, sid)
                    loss = training_loss(pred, y)

            scaler.scale(loss).backward()
            if cfg.grad_clip is not None:
                scaler.unscale_(optimizer)
                torch.nn.utils.clip_grad_norm_(model.parameters(), cfg.grad_clip)
            scaler.step(optimizer)
            scaler.update()
            scheduler.step()
            global_step += 1
            train_losses.append(float(loss.detach().cpu()))

        model.eval()
        val_losses = []
        with torch.no_grad():
            for x_raw, x_wav, y, sid in val_loader:
                x_raw = x_raw.to(DEVICE, non_blocking=True)
                x_wav = x_wav.to(DEVICE, non_blocking=True)
                y = y.to(DEVICE, non_blocking=True)
                sid = sid.to(DEVICE, non_blocking=True)
                pred = model_forward(model, x_raw, x_wav, sid)
                val_losses.append(float(training_loss(pred, y).detach().cpu()))

        val_loss = float(np.mean(val_losses)) if val_losses else float("inf")
        train_loss = float(np.mean(train_losses)) if train_losses else float("inf")
        if val_loss < best_val:
            best_val = val_loss
            best_state = {k: v.detach().cpu().clone() for k, v in model.state_dict().items()}
            patience = cfg.early_stop
        else:
            patience -= 1

        if epoch == 1 or epoch % 10 == 0:
            current_lr = optimizer.param_groups[0]["lr"]
            print(f"      Epoch {epoch:03d} | train={train_loss:.5f} | val={val_loss:.5f} | lr={current_lr:.2e}")

        if patience <= 0:
            print(f"      Early stopping at epoch {epoch}; best val={best_val:.5f}")
            break

    if best_state is not None:
        model.load_state_dict(best_state)
    return model


def predict_model(model: nn.Module, loader: DataLoader, y_scaler: StandardScaler, return_gate: bool = False):
    model.eval()
    preds, trues, gates = [], [], []
    with torch.no_grad():
        for x_raw, x_wav, y, sid in loader:
            x_raw = x_raw.to(DEVICE, non_blocking=True)
            x_wav = x_wav.to(DEVICE, non_blocking=True)
            sid = sid.to(DEVICE, non_blocking=True)
            if return_gate and isinstance(model, E3TSF):
                pred, gate, _ = model(x_raw, x_wav, sid, return_gate=True)
                gates.append(gate.detach().cpu().numpy())
            else:
                pred = model_forward(model, x_raw, x_wav, sid)
            preds.append(pred.detach().cpu().numpy())
            trues.append(y.numpy())

    pred_scaled = np.vstack(preds)
    true_scaled = np.vstack(trues)
    pred = y_scaler.inverse_transform(pred_scaled.reshape(-1, 1)).reshape(pred_scaled.shape)
    true = y_scaler.inverse_transform(true_scaled.reshape(-1, 1)).reshape(true_scaled.shape)
    if gates:
        return true, pred, np.vstack(gates)
    return true, pred, None


# ============================================================
# 8. Data preparation
# ============================================================

def prepare_single_data(dcfg: Dict):
    cfg = dcfg["config"]
    df, train_end_index, test_start_index = load_single_dataset(dcfg)
    y = df[dcfg["target_col"]].values.astype(np.float32)
    X, Y, T, SID = build_windows_single(y, cfg, stride=cfg.stride)
    val_start, test_start = chronological_split_indices(len(y), cfg.lookback, cfg.horizon, train_end_index, test_start_index)
    train_mask = T < val_start
    val_mask = (T >= val_start) & (T < test_start)
    test_mask = T >= test_start
    return df, y, X, Y, T, SID, train_mask, val_mask, test_mask, 1


def prepare_panel_data(dcfg: Dict):
    cfg = dcfg["config"]
    df, _, _, series_to_id = read_panel_train_test(dcfg)
    X, Y, T, SID, series_lengths = build_windows_panel(df, dcfg["target_col"], cfg)
    train_mask = np.zeros(len(T), dtype=bool)
    val_mask = np.zeros(len(T), dtype=bool)
    test_mask = np.zeros(len(T), dtype=bool)
    for sid, n in series_lengths.items():
        val_start, test_start = chronological_split_indices(n, cfg.lookback, cfg.horizon)
        m = SID == sid
        train_mask[m] = T[m] < val_start
        val_mask[m] = (T[m] >= val_start) & (T[m] < test_start)
        test_mask[m] = T[m] >= test_start
    y = None
    return df, y, X, Y, T, SID, train_mask, val_mask, test_mask, len(series_to_id)


def make_loaders(dcfg: Dict, seed: int = SEED):
    cfg = dcfg["config"]
    if dcfg.get("id_cols"):
        df, raw_y, X, Y, T, SID, train_mask, val_mask, test_mask, n_series = prepare_panel_data(dcfg)
        val_start, test_start = None, None
    else:
        df, raw_y, X, Y, T, SID, train_mask, val_mask, test_mask, n_series = prepare_single_data(dcfg)
        df_tmp, train_end_index, test_start_index = load_single_dataset(dcfg)
        val_start, test_start = chronological_split_indices(len(raw_y), cfg.lookback, cfg.horizon, train_end_index, test_start_index)

    if train_mask.sum() == 0 or val_mask.sum() == 0 or test_mask.sum() == 0:
        raise RuntimeError(
            f"Invalid split for {dcfg['name']}: train={train_mask.sum()}, val={val_mask.sum()}, test={test_mask.sum()}"
        )

    train_idx = np.where(train_mask)[0]
    val_idx = np.where(val_mask)[0]
    test_idx = np.where(test_mask)[0]
    train_idx = train_idx[maybe_sample_indices(len(train_idx), cfg.max_train_windows, seed)]
    val_idx = val_idx[maybe_sample_indices(len(val_idx), cfg.max_val_windows, seed + 1)]
    test_idx = test_idx[maybe_sample_indices(len(test_idx), cfg.max_test_windows, seed + 2)]

    Xtr, Xva, Xte = X[train_idx], X[val_idx], X[test_idx]
    Ytr, Yva, Yte = Y[train_idx], Y[val_idx], Y[test_idx]
    SIDtr, SIDva, SIDte = SID[train_idx], SID[val_idx], SID[test_idx]
    Tte = T[test_idx]

    x_scaler = StandardScaler().fit(Xtr.reshape(-1, 1))
    y_scaler = StandardScaler().fit(Ytr.reshape(-1, 1))

    def sx(a): return x_scaler.transform(a.reshape(-1, 1)).reshape(a.shape).astype(np.float32)
    def sy(a): return y_scaler.transform(a.reshape(-1, 1)).reshape(a.shape).astype(np.float32)

    Xtr_s, Xva_s, Xte_s = sx(Xtr), sx(Xva), sx(Xte)
    Ytr_s, Yva_s, Yte_s = sy(Ytr), sy(Yva), sy(Yte)

    print("    Building wavelet tensors window-by-window; this avoids future leakage.")
    Xtr_w = build_wavelet_tensor(Xtr_s, cfg)
    Xva_w = build_wavelet_tensor(Xva_s, cfg)
    Xte_w = build_wavelet_tensor(Xte_s, cfg)
    n_scales = int(Xtr_w.shape[1])

    train_ds = ForecastDataset(Xtr_s, Xtr_w, Ytr_s, SIDtr)
    val_ds = ForecastDataset(Xva_s, Xva_w, Yva_s, SIDva)
    test_ds = ForecastDataset(Xte_s, Xte_w, Yte_s, SIDte)

    train_loader = DataLoader(train_ds, batch_size=cfg.batch_size, shuffle=True, num_workers=cfg.num_workers, pin_memory=(DEVICE == "cuda"))
    val_loader = DataLoader(val_ds, batch_size=cfg.batch_size, shuffle=False, num_workers=cfg.num_workers, pin_memory=(DEVICE == "cuda"))
    test_loader = DataLoader(test_ds, batch_size=cfg.batch_size, shuffle=False, num_workers=cfg.num_workers, pin_memory=(DEVICE == "cuda"))

    info = {
        "df": df,
        "raw_y": raw_y,
        "x_scaler": x_scaler,
        "y_scaler": y_scaler,
        "n_series": n_series,
        "n_scales": n_scales,
        "n_train_windows": len(train_idx),
        "n_val_windows": len(val_idx),
        "n_test_windows": len(test_idx),
        "train_idx": train_idx,
        "val_idx": val_idx,
        "test_idx": test_idx,
        "val_start": val_start,
        "test_start": test_start,
        "T_test": Tte,
        "Y_test_original": Yte,
        "SID_test": SIDte,
        "example_x_original": Xte[0].copy() if len(Xte) else None,
        "example_x_raw_scaled": Xte_s[0].copy() if len(Xte_s) else None,
        "example_x_wav": Xte_w[0].copy() if len(Xte_w) else None,
        "example_y_original": Yte[0].copy() if len(Yte) else None,
        "is_panel": bool(dcfg.get("id_cols")),
    }
    return train_loader, val_loader, test_loader, y_scaler, info


# ============================================================
# 9. Seasonal naive for single and panel
# ============================================================

def evaluate_seasonal_naive_single(raw_y: np.ndarray, T_test: np.ndarray, horizon: int, period: int) -> Tuple[np.ndarray, np.ndarray]:
    preds, trues = [], []
    for t in T_test:
        history = raw_y[:t]
        pred = seasonal_naive_forecast(history, horizon, period)
        true = raw_y[t:t + horizon]
        if len(true) == horizon:
            preds.append(pred)
            trues.append(true)
    return np.asarray(trues), np.asarray(preds)


def evaluate_seasonal_naive_panel(dcfg: Dict, info: Dict, horizon: int, period: int) -> Tuple[np.ndarray, np.ndarray]:
    df = info["df"]
    ycol = dcfg["target_col"]
    preds, trues = [], []
    series_cache = {int(sid): g[ycol].values.astype(np.float32) for sid, g in df.groupby("series_id", sort=False)}
    for sid, t, true in zip(info["SID_test"], info["T_test"], info["Y_test_original"]):
        y = series_cache[int(sid)]
        pred = seasonal_naive_forecast(y[:t], horizon, period)
        preds.append(pred)
        trues.append(true)
    return np.asarray(trues), np.asarray(preds)


def dataset_config_for_metadata(dcfg: Dict) -> Dict:
    out = {k: v for k, v in dcfg.items() if k != "config"}
    out["config"] = asdict(dcfg["config"])
    return out


def save_run_metadata():
    metadata = {
        "run_type": "final_kaggle_four_dataset_run",
        "output_directory": OUT_DIR,
        "device": DEVICE,
        "run_seeds": RUN_SEEDS,
        "python": sys.version,
        "platform": platform.platform(),
        "numpy_version": np.__version__,
        "pandas_version": pd.__version__,
        "sklearn_version": sklearn.__version__,
        "torch_version": torch.__version__,
        "cuda_available": bool(torch.cuda.is_available()),
        "cuda_device_name": torch.cuda.get_device_name(0) if torch.cuda.is_available() else None,
        "datasets": [dataset_config_for_metadata(d) for d in DATASETS],
    }
    path = os.path.join(METADATA_DIR, "run_metadata.json")
    with open(path, "w", encoding="utf-8") as f:
        json.dump(metadata, f, indent=2)
    print(f"Saved run metadata: {path}")


def save_scalers(dataset_name: str, seed: int, info: Dict):
    payload = {
        "x_scaler": info.get("x_scaler"),
        "y_scaler": info.get("y_scaler"),
    }
    path = os.path.join(METADATA_DIR, f"{_safe_dataset_name(dataset_name)}_seed{seed}_scalers.pkl")
    with open(path, "wb") as f:
        pickle.dump(payload, f)
    print(f"    Saved scalers: {path}")


def save_split_report(dataset_name: str, seed: int, info: Dict):
    rows = []
    for split, idx in [("train", info.get("train_idx")), ("validation", info.get("val_idx")), ("test", info.get("test_idx"))]:
        idx = np.asarray(idx if idx is not None else [], dtype=np.int64)
        rows.append({
            "dataset": dataset_name,
            "seed": seed,
            "split": split,
            "n_windows": int(len(idx)),
            "first_window_index": int(idx.min()) if len(idx) else None,
            "last_window_index": int(idx.max()) if len(idx) else None,
        })
    path = os.path.join(METADATA_DIR, f"{_safe_dataset_name(dataset_name)}_seed{seed}_split_summary.csv")
    pd.DataFrame(rows).to_csv(path, index=False)
    print(f"    Saved split report: {path}")


def save_model_checkpoint(model: nn.Module, dataset_name: str, seed: int, model_name: str, cfg: ModelConfig, metrics: Dict[str, float]):
    safe_dataset = _safe_dataset_name(dataset_name)
    safe_model = _safe_dataset_name(model_name)
    state_dict = {k: v.detach().cpu() for k, v in model.state_dict().items()}
    payload = {
        "dataset": dataset_name,
        "seed": int(seed),
        "model_name": model_name,
        "config": asdict(cfg),
        "metrics": {k: float(v) for k, v in metrics.items()},
        "state_dict": state_dict,
    }
    path = os.path.join(CHECKPOINT_DIR, f"{safe_dataset}_seed{seed}_{safe_model}.pt")
    torch.save(payload, path)
    print(f"    Saved checkpoint: {path}")


def save_dataset_profile_plot(dcfg: Dict, info: Dict, seed: int):
    name = dcfg["name"]
    cfg = dcfg["config"]
    ycol = dcfg["target_col"]
    if info["is_panel"]:
        df = info["df"]
        sid = int(df["series_id"].iloc[0])
        g = df[df["series_id"] == sid].reset_index(drop=True)
        y = g[ycol].values.astype(np.float32)
        val_start, test_start = chronological_split_indices(len(y), cfg.lookback, cfg.horizon)
        subtitle = f"sample series_id={sid}"
    else:
        y = np.asarray(info["raw_y"], dtype=np.float32)
        val_start, test_start = info.get("val_start"), info.get("test_start")
        subtitle = "single series"

    if val_start is None or test_start is None:
        val_start, test_start = chronological_split_indices(len(y), cfg.lookback, cfg.horizon)

    split = np.full(len(y), "train", dtype=object)
    split[int(val_start):int(test_start)] = "validation"
    split[int(test_start):] = "test"
    src = pd.DataFrame({"time_index": np.arange(len(y)), "value": y, "split": split})
    src_path = os.path.join(FIGURE_SOURCE_DIR, f"{_safe_dataset_name(name)}_seed{seed}_dataset_profile_source.csv")
    src.to_csv(src_path, index=False)

    if not HAS_MPL:
        return
    plt.figure(figsize=(10, 4.6))
    plt.plot(src["time_index"], src["value"], color="#2f5f9e", linewidth=1.1)
    plt.axvline(val_start, color="#d08a2d", linestyle="--", linewidth=1.0, label="validation starts")
    plt.axvline(test_start, color="#c9475d", linestyle="--", linewidth=1.0, label="test starts")
    plt.title(f"{name}: chronological split profile ({subtitle})")
    plt.xlabel("Time index")
    plt.ylabel(ycol)
    plt.legend()
    plt.tight_layout()
    path = os.path.join(PAPER_FIG_DIR, f"{_safe_dataset_name(name)}_seed{seed}_chronological_split_profile.png")
    plt.savefig(path, dpi=300, bbox_inches="tight")
    plt.close()
    print(f"    Saved chronological split profile: {path}")


def save_example_window_plot(dcfg: Dict, info: Dict, seed: int):
    name = dcfg["name"]
    ycol = dcfg["target_col"]
    x = info.get("example_x_original")
    y = info.get("example_y_original")
    if x is None or y is None:
        return
    x = np.asarray(x, dtype=np.float32)
    y = np.asarray(y, dtype=np.float32)
    src = pd.DataFrame({
        "relative_index": np.concatenate([np.arange(-len(x), 0), np.arange(1, len(y) + 1)]),
        "value": np.concatenate([x, y]),
        "segment": ["lookback"] * len(x) + ["forecast_target"] * len(y),
    })
    src_path = os.path.join(FIGURE_SOURCE_DIR, f"{_safe_dataset_name(name)}_seed{seed}_example_window_source.csv")
    src.to_csv(src_path, index=False)

    if not HAS_MPL:
        return
    plt.figure(figsize=(9.5, 4.6))
    plt.plot(np.arange(-len(x), 0), x, label="Lookback window", color="#315f9c", linewidth=1.1)
    plt.plot(np.arange(1, len(y) + 1), y, label="Forecast target", color="#c9475d", linewidth=1.1)
    plt.axvline(0, color="#555555", linewidth=0.9)
    plt.title(f"{name}: example input-target forecasting window")
    plt.xlabel("Relative time index")
    plt.ylabel(ycol)
    plt.legend()
    plt.tight_layout()
    path = os.path.join(PAPER_FIG_DIR, f"{_safe_dataset_name(name)}_seed{seed}_example_input_target_window.png")
    plt.savefig(path, dpi=300, bbox_inches="tight")
    plt.close()
    print(f"    Saved example input-target window: {path}")


def save_wavelet_component_plot(dcfg: Dict, info: Dict, seed: int):
    name = dcfg["name"]
    x_wav = info.get("example_x_wav")
    if x_wav is None:
        return
    x_wav = np.asarray(x_wav, dtype=np.float32)
    rows = []
    for scale_idx in range(x_wav.shape[0]):
        scale_name = "approximation" if scale_idx == 0 else f"detail_{scale_idx}"
        for t, value in enumerate(x_wav[scale_idx]):
            rows.append({"scale_index": scale_idx, "scale_name": scale_name, "time_index": t, "value_scaled": float(value)})
    src = pd.DataFrame(rows)
    src_path = os.path.join(FIGURE_SOURCE_DIR, f"{_safe_dataset_name(name)}_seed{seed}_wavelet_components_source.csv")
    src.to_csv(src_path, index=False)

    if not HAS_MPL:
        return
    n_scales = x_wav.shape[0]
    fig, axes = plt.subplots(n_scales, 1, figsize=(9.5, max(2.0, 1.65 * n_scales)), sharex=True)
    axes = np.asarray(axes).reshape(-1)
    colors = ["#315f9c", "#7a5ab5", "#c9475d", "#56a36a", "#d08a2d"]
    for i, ax in enumerate(axes):
        label = "Approximation" if i == 0 else f"Detail {i}"
        ax.plot(x_wav[i], color=colors[i % len(colors)], linewidth=1.0)
        ax.set_ylabel(label)
        ax.grid(True, alpha=0.18)
    axes[-1].set_xlabel("Lookback time index")
    fig.suptitle(f"{name}: window-level wavelet components", y=1.01)
    fig.tight_layout()
    path = os.path.join(PAPER_FIG_DIR, f"{_safe_dataset_name(name)}_seed{seed}_wavelet_components.png")
    fig.savefig(path, dpi=300, bbox_inches="tight")
    plt.close(fig)
    print(f"    Saved wavelet component plot: {path}")


def save_horizon_error_plot(dataset_name: str, seed: int, y_true: np.ndarray, y_pred: np.ndarray):
    y_true = np.asarray(y_true)
    y_pred = np.asarray(y_pred)
    if y_true.ndim != 2 or y_pred.ndim != 2:
        return
    rows = []
    for h in range(y_true.shape[1]):
        rows.append({
            "horizon": h + 1,
            "MAE": float(mean_absolute_error(y_true[:, h], y_pred[:, h])),
            "RMSE": float(math.sqrt(mean_squared_error(y_true[:, h], y_pred[:, h]))),
            "sMAPE": float(np.mean(2.0 * np.abs(y_true[:, h] - y_pred[:, h]) / (np.abs(y_true[:, h]) + np.abs(y_pred[:, h]) + 1e-8)) * 100),
        })
    src = pd.DataFrame(rows)
    src_path = os.path.join(FIGURE_SOURCE_DIR, f"{_safe_dataset_name(dataset_name)}_seed{seed}_horizon_error_source.csv")
    src.to_csv(src_path, index=False)

    if not HAS_MPL:
        return
    fig, axes = plt.subplots(3, 1, figsize=(8.5, 8.5), sharex=True)
    for ax, metric, color in zip(axes, ["MAE", "RMSE", "sMAPE"], ["#315f9c", "#c9475d", "#56a36a"]):
        ax.plot(src["horizon"], src[metric], marker="o", color=color)
        ax.set_ylabel(metric)
        ax.grid(True, alpha=0.22)
    axes[-1].set_xlabel("Forecast horizon")
    fig.suptitle(f"{dataset_name}: E3-TSF horizon-wise error", y=1.01)
    fig.tight_layout()
    path = os.path.join(PAPER_FIG_DIR, f"{_safe_dataset_name(dataset_name)}_seed{seed}_horizon_wise_error.png")
    fig.savefig(path, dpi=300, bbox_inches="tight")
    plt.close(fig)
    print(f"    Saved horizon-wise error plot: {path}")


def save_residual_scatter_plot(dataset_name: str, seed: int, y_true: np.ndarray, y_pred: np.ndarray):
    yt = np.asarray(y_true).reshape(-1)
    yp = np.asarray(y_pred).reshape(-1)
    residual = yt - yp
    src = pd.DataFrame({"y_true": yt, "y_pred": yp, "residual": residual})
    src_path = os.path.join(FIGURE_SOURCE_DIR, f"{_safe_dataset_name(dataset_name)}_seed{seed}_residual_scatter_source.csv")
    src.to_csv(src_path, index=False)

    if not HAS_MPL:
        return
    sample_n = min(5000, len(src))
    src_plot = src.sample(n=sample_n, random_state=seed) if len(src) > sample_n else src
    plt.figure(figsize=(6.2, 5.4))
    plt.scatter(src_plot["y_true"], src_plot["y_pred"], s=8, alpha=0.35, color="#315f9c")
    lo = float(min(src_plot["y_true"].min(), src_plot["y_pred"].min()))
    hi = float(max(src_plot["y_true"].max(), src_plot["y_pred"].max()))
    plt.plot([lo, hi], [lo, hi], color="#c9475d", linewidth=1.2)
    plt.title(f"{dataset_name}: E3-TSF predicted vs actual")
    plt.xlabel("Actual")
    plt.ylabel("Predicted")
    plt.tight_layout()
    path = os.path.join(PAPER_FIG_DIR, f"{_safe_dataset_name(dataset_name)}_seed{seed}_predicted_vs_actual_scatter.png")
    plt.savefig(path, dpi=300, bbox_inches="tight")
    plt.close()
    print(f"    Saved predicted-vs-actual scatter: {path}")


def save_gate_distribution_plot(dataset_name: str, seed: int, gates: Optional[np.ndarray]):
    if gates is None or len(gates) == 0:
        return
    gate_names = ["Raw patch", "Wavelet cross-scale", "Trend-seasonal"]
    g = np.asarray(gates)
    long_rows = []
    for i, name in enumerate(gate_names):
        for j, value in enumerate(g[:, i]):
            long_rows.append({"test_window_index": j, "expert": name, "gate_weight": float(value)})
    src = pd.DataFrame(long_rows)
    src_path = os.path.join(FIGURE_SOURCE_DIR, f"{_safe_dataset_name(dataset_name)}_seed{seed}_gate_distribution_source.csv")
    src.to_csv(src_path, index=False)

    if not HAS_MPL:
        return
    plt.figure(figsize=(7.5, 4.8))
    plt.boxplot([g[:, i] for i in range(g.shape[1])], labels=gate_names, showfliers=False)
    plt.title(f"{dataset_name}: distribution of dynamic gate weights")
    plt.ylabel("Gate weight")
    plt.xticks(rotation=15, ha="right")
    plt.tight_layout()
    path = os.path.join(PAPER_FIG_DIR, f"{_safe_dataset_name(dataset_name)}_seed{seed}_gate_weight_distribution.png")
    plt.savefig(path, dpi=300, bbox_inches="tight")
    plt.close()
    print(f"    Saved gate-weight distribution plot: {path}")


# ============================================================
# 10. Publication figure helpers
# ============================================================


def _safe_dataset_name(name: str) -> str:
    return "".join(ch if ch.isalnum() or ch in "-_" else "_" for ch in str(name))


def save_mae_model_comparison_bar(result_df: pd.DataFrame, dataset_name: str, seed: int):
    if result_df is None or len(result_df) == 0:
        return
    df = result_df.sort_values("MAE", ascending=True)
    src_path = os.path.join(FIGURE_SOURCE_DIR, f"{_safe_dataset_name(dataset_name)}_seed{seed}_MAE_model_comparison_source.csv")
    df.to_csv(src_path, index=False)
    if not HAS_MPL:
        return
    plt.figure(figsize=(8, 4.8))
    plt.bar(df["model"], df["MAE"])
    plt.title(f"{dataset_name}: MAE model comparison")
    plt.xlabel("Model")
    plt.ylabel("MAE")
    plt.xticks(rotation=30, ha="right")
    plt.tight_layout()
    path = os.path.join(PAPER_FIG_DIR, f"{_safe_dataset_name(dataset_name)}_seed{seed}_MAE_model_comparison.png")
    plt.savefig(path, dpi=300)
    plt.close()
    print(f"    Saved MAE model comparison graph: {path}")


def save_error_distribution_plot(dataset_name: str, seed: int, y_true: np.ndarray, y_pred: np.ndarray):
    err = (np.asarray(y_true).reshape(-1) - np.asarray(y_pred).reshape(-1))
    src_path = os.path.join(FIGURE_SOURCE_DIR, f"{_safe_dataset_name(dataset_name)}_seed{seed}_error_distribution_source.csv")
    pd.DataFrame({"forecast_error_actual_minus_predicted": err}).to_csv(src_path, index=False)
    if not HAS_MPL:
        return
    plt.figure(figsize=(7.5, 4.8))
    plt.hist(err, bins=50)
    plt.title(f"{dataset_name}: E3-TSF error distribution")
    plt.xlabel("Forecast error: actual - predicted")
    plt.ylabel("Frequency")
    plt.tight_layout()
    path = os.path.join(PAPER_FIG_DIR, f"{_safe_dataset_name(dataset_name)}_seed{seed}_E3TSF_error_distribution.png")
    plt.savefig(path, dpi=300)
    plt.close()
    print(f"    Saved error distribution graph: {path}")


def save_dynamic_gate_weight_visualization(dataset_name: str, seed: int, gates: Optional[np.ndarray]):
    if gates is None or len(gates) == 0:
        return
    gate_names = ["Raw patch", "Wavelet cross-scale", "Trend-seasonal"]
    g = np.asarray(gates)
    mean_df = pd.DataFrame({"expert": gate_names, "mean_gate_weight": g.mean(axis=0)})
    mean_src_path = os.path.join(FIGURE_SOURCE_DIR, f"{_safe_dataset_name(dataset_name)}_seed{seed}_average_gate_weights_source.csv")
    mean_df.to_csv(mean_src_path, index=False)
    if not HAS_MPL:
        return

    plt.figure(figsize=(7.5, 4.8))
    plt.bar(gate_names, g.mean(axis=0))
    plt.title(f"{dataset_name}: average dynamic gate weights")
    plt.xlabel("Expert branch")
    plt.ylabel("Mean gate weight")
    plt.xticks(rotation=20, ha="right")
    plt.tight_layout()
    path = os.path.join(PAPER_FIG_DIR, f"{_safe_dataset_name(dataset_name)}_seed{seed}_E3TSF_average_gate_weights.png")
    plt.savefig(path, dpi=300)
    plt.close()
    print(f"    Saved dynamic gate bar graph: {path}")

    sample_n = min(300, len(g))
    plt.figure(figsize=(9, 4.8))
    for i, label in enumerate(gate_names):
        plt.plot(g[:sample_n, i], label=label)
    plt.title(f"{dataset_name}: dynamic gate weights across test windows")
    plt.xlabel("Test window index")
    plt.ylabel("Gate weight")
    plt.legend()
    plt.tight_layout()
    path = os.path.join(PAPER_FIG_DIR, f"{_safe_dataset_name(dataset_name)}_seed{seed}_E3TSF_dynamic_gate_weights.png")
    plt.savefig(path, dpi=300)
    plt.close()
    print(f"    Saved dynamic gate line graph: {path}")


def select_horizon_indices(horizon: int, max_points: int = 4) -> List[int]:
    if horizon <= 0:
        return []
    raw = [0, horizon // 3, (2 * horizon) // 3, horizon - 1]
    out = []
    for h in raw:
        h = int(min(max(h, 0), horizon - 1))
        if h not in out:
            out.append(h)
    return out[:max_points]


def save_actual_vs_predicted_horizon_plots(dataset_name: str, seed: int, y_true: np.ndarray, y_pred: np.ndarray, target_col: str):
    y_true = np.asarray(y_true)
    y_pred = np.asarray(y_pred)
    if y_true.ndim != 2 or y_pred.ndim != 2:
        return
    H = y_true.shape[1]
    horizons = select_horizon_indices(H, max_points=4)
    plot_len = min(300, y_true.shape[0])
    rows = []
    for h in horizons:
        for i in range(plot_len):
            rows.append({"test_window_index": i, "horizon": h + 1, "y_true": y_true[i, h], "y_pred": y_pred[i, h]})
    src_path = os.path.join(FIGURE_SOURCE_DIR, f"{_safe_dataset_name(dataset_name)}_seed{seed}_actual_vs_predicted_horizons_source.csv")
    pd.DataFrame(rows).to_csv(src_path, index=False)

    if not HAS_MPL:
        return

    # Single combined figure with 3-4 future horizons.
    n = len(horizons)
    rows = 2 if n > 2 else 1
    cols = 2 if n > 1 else 1
    fig, axes = plt.subplots(rows, cols, figsize=(12, 7 if rows == 2 else 4.8))
    axes = np.asarray(axes).reshape(-1)
    for ax, h in zip(axes, horizons):
        ax.plot(y_true[:plot_len, h], label="Actual")
        ax.plot(y_pred[:plot_len, h], label="Predicted")
        ax.set_title(f"Horizon t+{h+1}")
        ax.set_xlabel("Test window index")
        ax.set_ylabel(target_col)
        ax.legend()
    for ax in axes[len(horizons):]:
        ax.axis("off")
    fig.suptitle(f"{dataset_name}: actual vs predicted across future horizons", y=1.02)
    fig.tight_layout()
    path = os.path.join(PAPER_FIG_DIR, f"{_safe_dataset_name(dataset_name)}_seed{seed}_E3TSF_actual_vs_predicted_multi_horizon.png")
    fig.savefig(path, dpi=300, bbox_inches="tight")
    plt.close(fig)
    print(f"    Saved actual vs predicted multi-horizon graph: {path}")

    # Also save individual horizon figures for paper use.
    for h in horizons:
        plt.figure(figsize=(8, 4.8))
        plt.plot(y_true[:plot_len, h], label="Actual")
        plt.plot(y_pred[:plot_len, h], label="Predicted")
        plt.title(f"{dataset_name}: actual vs predicted at horizon t+{h+1}")
        plt.xlabel("Test window index")
        plt.ylabel(target_col)
        plt.legend()
        plt.tight_layout()
        path = os.path.join(PAPER_FIG_DIR, f"{_safe_dataset_name(dataset_name)}_seed{seed}_E3TSF_actual_vs_predicted_h{h+1}.png")
        plt.savefig(path, dpi=300)
        plt.close()


def save_prediction_csvs(dataset_name: str, seed: int, y_true: np.ndarray, y_pred: np.ndarray):
    pred_flat_df = pd.DataFrame({
        "y_true_flat": np.asarray(y_true).reshape(-1),
        "y_pred_flat": np.asarray(y_pred).reshape(-1),
    })
    pred_path = os.path.join(TABLE_DIR, f"{_safe_dataset_name(dataset_name)}_seed{seed}_E3TSF_predictions_flat.csv")
    pred_flat_df.to_csv(pred_path, index=False)

    rows = []
    y_true = np.asarray(y_true)
    y_pred = np.asarray(y_pred)
    for i in range(y_true.shape[0]):
        for h in range(y_true.shape[1]):
            rows.append({"sample_index": i, "horizon": h + 1, "y_true": y_true[i, h], "y_pred": y_pred[i, h]})
    pred_long_df = pd.DataFrame(rows)
    pred_long_path = os.path.join(TABLE_DIR, f"{_safe_dataset_name(dataset_name)}_seed{seed}_E3TSF_predictions_by_horizon.csv")
    pred_long_df.to_csv(pred_long_path, index=False)
    print(f"    Saved predictions: {pred_path}")
    print(f"    Saved horizon-wise predictions: {pred_long_path}")

# ============================================================
# 10. Experiment runner
# ============================================================

def run_single_dataset(dcfg: Dict, seed: int = SEED) -> pd.DataFrame:
    name = dcfg["name"]
    cfg: ModelConfig = dcfg["config"]
    print("\n" + "=" * 100)
    print(f"Dataset: {name}")
    print(f"Device: {DEVICE}")
    print(f"Seed: {seed}")
    print(f"Config: {asdict(cfg)}")

    set_seed(seed)
    train_loader, val_loader, test_loader, y_scaler, info = make_loaders(dcfg, seed=seed)
    n_scales = info["n_scales"]
    n_series = info["n_series"]
    print(f"    Series count: {n_series}")
    print(f"    Windows: train={info['n_train_windows']}, val={info['n_val_windows']}, test={info['n_test_windows']}")
    print(f"    Wavelet scales: {n_scales}")

    setup_outputs = [
        ("scalers", lambda: save_scalers(name, seed, info)),
        ("split report", lambda: save_split_report(name, seed, info)),
        ("chronological split profile", lambda: save_dataset_profile_plot(dcfg, info, seed)),
        ("example input-target window", lambda: save_example_window_plot(dcfg, info, seed)),
        ("wavelet component plot", lambda: save_wavelet_component_plot(dcfg, info, seed)),
    ]
    for label, save_fn in setup_outputs:
        try:
            save_fn()
        except Exception as e:
            print(f"    Warning: could not save {label}: {repr(e)}")

    results = []

    # Seasonal naive baseline timing.
    sn_start = time.perf_counter()
    if info["is_panel"]:
        y_true_sn, y_pred_sn = evaluate_seasonal_naive_panel(dcfg, info, cfg.horizon, cfg.seasonal_period)
    else:
        y_true_sn, y_pred_sn = evaluate_seasonal_naive_single(info["raw_y"], info["T_test"], cfg.horizon, cfg.seasonal_period)
    m_sn = compute_metrics(y_true_sn, y_pred_sn)
    sn_total = time.perf_counter() - sn_start
    results.append({"dataset": name, "seed": seed, "model": "SeasonalNaive", **m_sn,
                    "train_seconds": 0.0, "inference_seconds": sn_total, "total_seconds": sn_total})
    print(f"    SeasonalNaive: MAE={m_sn['MAE']:.4f}, RMSE={m_sn['RMSE']:.4f}, RRSE={m_sn['RRSE']:.4f}, CORR={m_sn['CORR']:.4f}, total_time={sn_total:.2f}s")

    models = {
        "DLinear": DLinearBaseline(cfg, n_series=n_series),
        "PatchTST": PatchTSTBaseline(cfg, n_series=n_series),
        "E3TSF_NoGateAvg": E3TSFNoGateAverage(n_scales, cfg, n_series=n_series),
        "E3TSF_DynamicGated": E3TSF(n_scales, cfg, n_series=n_series),
    }

    for model_name, model in models.items():
        print(f"\n    Training {model_name}...")
        set_seed(seed)
        model = model.to(DEVICE)

        sync_device()
        train_start = time.perf_counter()
        model = train_model(model, train_loader, val_loader, cfg)
        sync_device()
        train_seconds = time.perf_counter() - train_start

        sync_device()
        infer_start = time.perf_counter()
        y_true, y_pred, gates = predict_model(model, test_loader, y_scaler, return_gate=(model_name == "E3TSF_DynamicGated"))
        sync_device()
        inference_seconds = time.perf_counter() - infer_start
        total_seconds = train_seconds + inference_seconds

        metrics = compute_metrics(y_true, y_pred)
        results.append({"dataset": name, "seed": seed, "model": model_name, **metrics,
                        "train_seconds": train_seconds, "inference_seconds": inference_seconds, "total_seconds": total_seconds})
        print(
            f"    {model_name}: MAE={metrics['MAE']:.4f}, RMSE={metrics['RMSE']:.4f}, "
            f"RRSE={metrics['RRSE']:.4f}, CORR={metrics['CORR']:.4f}, "
            f"train_time={train_seconds:.2f}s, inference_time={inference_seconds:.2f}s, total_time={total_seconds:.2f}s"
        )
        save_model_checkpoint(model, name, seed, model_name, cfg, metrics)

        if model_name == "E3TSF_DynamicGated":
            save_prediction_csvs(name, seed, y_true, y_pred)
            save_actual_vs_predicted_horizon_plots(name, seed, y_true, y_pred, dcfg["target_col"])
            save_error_distribution_plot(name, seed, y_true, y_pred)
            save_horizon_error_plot(name, seed, y_true, y_pred)
            save_residual_scatter_plot(name, seed, y_true, y_pred)

            if gates is not None:
                gate_df = pd.DataFrame(gates, columns=["gate_raw_patch", "gate_wavelet_cross_scale", "gate_trend_seasonal"])
                gate_path = os.path.join(TABLE_DIR, f"{_safe_dataset_name(name)}_seed{seed}_E3TSF_gate_weights.csv")
                gate_df.to_csv(gate_path, index=False)
                print(f"    Saved dynamic gate weights: {gate_path}")
                print("    Average gate weights:", gate_df.mean().to_dict())
                save_dynamic_gate_weight_visualization(name, seed, gates)
                save_gate_distribution_plot(name, seed, gates)

            # Keep one simple flattened actual-vs-predicted graph as an additional quick diagnostic.
            if HAS_MPL:
                plot_len = min(500, y_true.size)
                plt.figure(figsize=(10, 4))
                plt.plot(y_true.reshape(-1)[:plot_len], label="Actual")
                plt.plot(y_pred.reshape(-1)[:plot_len], label="Predicted")
                plt.title(f"{name}: E3-TSF forecast vs actual")
                plt.xlabel("Flattened forecast points")
                plt.ylabel(dcfg["target_col"])
                plt.legend()
                plt.tight_layout()
                fig_path = os.path.join(PAPER_FIG_DIR, f"{_safe_dataset_name(name)}_seed{seed}_E3TSF_actual_vs_predicted_flat.png")
                plt.savefig(fig_path, dpi=300)
                plt.close()
                print(f"    Saved flat actual vs predicted graph: {fig_path}")

        del model
        if torch.cuda.is_available():
            torch.cuda.empty_cache()

    result_df = pd.DataFrame(results)
    result_df = result_df.sort_values("MAE").reset_index(drop=True)
    result_path = os.path.join(TABLE_DIR, f"{_safe_dataset_name(name)}_seed{seed}_results.csv")
    result_df.to_csv(result_path, index=False)
    save_mae_model_comparison_bar(result_df, name, seed)
    print(f"\n    Saved dataset results: {result_path}")
    return result_df

def dataset_paths(dcfg: Dict) -> List[str]:
    if dcfg.get("id_cols"):
        return [dcfg["train_path"]]
    if "train_path" in dcfg and "test_path" in dcfg:
        return [dcfg["train_path"], dcfg["test_path"]]
    return [dcfg["single_path"]]


def summarize_mean_std(summary: pd.DataFrame) -> pd.DataFrame:
    metric_cols = ["MAE", "RMSE", "RRSE", "CORR", "R2", "sMAPE", "WAPE", "train_seconds", "inference_seconds", "total_seconds"]
    rows = []
    for (dataset, model), g in summary.groupby(["dataset", "model"], sort=False):
        row = {"dataset": dataset, "model": model, "runs": int(g["seed"].nunique())}
        for m in metric_cols:
            row[f"{m}_mean"] = float(g[m].mean())
            row[f"{m}_std"] = float(g[m].std(ddof=0))
        rows.append(row)
    out = pd.DataFrame(rows)
    return out.sort_values(["dataset", "MAE_mean"]).reset_index(drop=True)


def save_overall_summary_plots(mean_std: pd.DataFrame, best_rows: pd.DataFrame, win_count: pd.DataFrame):
    if mean_std is None or len(mean_std) == 0:
        return

    mean_src = os.path.join(FIGURE_SOURCE_DIR, "overall_mean_std_summary_source.csv")
    mean_std.to_csv(mean_src, index=False)
    if best_rows is not None and len(best_rows) > 0:
        best_src = os.path.join(FIGURE_SOURCE_DIR, "overall_best_model_per_dataset_source.csv")
        best_rows.to_csv(best_src, index=False)
    if win_count is not None and len(win_count) > 0:
        win_src = os.path.join(FIGURE_SOURCE_DIR, "overall_mae_win_count_source.csv")
        win_count.to_csv(win_src, index=False)

    if not HAS_MPL:
        return

    pivot = mean_std.pivot(index="dataset", columns="model", values="MAE_mean")
    if len(pivot) > 0:
        fig, ax = plt.subplots(figsize=(10.5, 4.8))
        arr = pivot.values.astype(float)
        im = ax.imshow(arr, aspect="auto", cmap="viridis")
        ax.set_xticks(np.arange(len(pivot.columns)))
        ax.set_xticklabels(pivot.columns, rotation=30, ha="right")
        ax.set_yticks(np.arange(len(pivot.index)))
        ax.set_yticklabels(pivot.index)
        ax.set_title("Mean MAE by dataset and model")
        for i in range(arr.shape[0]):
            for j in range(arr.shape[1]):
                if np.isfinite(arr[i, j]):
                    ax.text(j, i, f"{arr[i, j]:.2f}", ha="center", va="center", color="white", fontsize=8)
        fig.colorbar(im, ax=ax, fraction=0.035, pad=0.02, label="MAE")
        fig.tight_layout()
        path = os.path.join(PAPER_FIG_DIR, "Overall_mean_MAE_heatmap.png")
        fig.savefig(path, dpi=300, bbox_inches="tight")
        plt.close(fig)
        print(f"Saved overall MAE heatmap: {path}")

    if best_rows is not None and len(best_rows) > 0:
        fig, ax = plt.subplots(figsize=(9.5, 4.8))
        bars = ax.bar(best_rows["dataset"], best_rows["MAE_mean"], color="#315f9c")
        ax.set_title("Best model per dataset based on mean MAE")
        ax.set_xlabel("Dataset")
        ax.set_ylabel("Mean MAE")
        ax.tick_params(axis="x", rotation=20)
        y_max = float(best_rows["MAE_mean"].max()) if len(best_rows) else 1.0
        for bar, model_name in zip(bars, best_rows["model"]):
            ax.text(
                bar.get_x() + bar.get_width() / 2,
                bar.get_height() + 0.02 * max(y_max, 1e-8),
                str(model_name),
                ha="center",
                va="bottom",
                fontsize=8,
                rotation=20,
            )
        fig.tight_layout()
        path = os.path.join(PAPER_FIG_DIR, "Overall_best_model_per_dataset_MAE.png")
        fig.savefig(path, dpi=300, bbox_inches="tight")
        plt.close(fig)
        print(f"Saved best-model MAE plot: {path}")

    if win_count is not None and len(win_count) > 0:
        pivot_win = win_count.pivot(index="dataset", columns="model", values="MAE_win_count").fillna(0)
        fig, ax = plt.subplots(figsize=(9.5, 4.8))
        pivot_win.plot(kind="bar", stacked=True, ax=ax)
        ax.set_title("MAE win count across seeds")
        ax.set_xlabel("Dataset")
        ax.set_ylabel("Number of seed-level wins")
        ax.tick_params(axis="x", rotation=20)
        ax.legend(title="Model", bbox_to_anchor=(1.02, 1), loc="upper left")
        fig.tight_layout()
        path = os.path.join(PAPER_FIG_DIR, "Overall_MAE_win_count_by_seed.png")
        fig.savefig(path, dpi=300, bbox_inches="tight")
        plt.close(fig)
        print(f"Saved MAE win-count plot: {path}")


# ============================================================
# 11. Figure 8-style input-length experiment
# ============================================================

RUN_INPUT_LENGTH_EXPERIMENT = True
INPUT_LENGTHS_FOR_FIG8 = [10, 30, 60, 90]


def _cfg_for_input_length(base_cfg: ModelConfig, lookback: int) -> ModelConfig:
    cfg = copy.deepcopy(base_cfg)
    cfg.lookback = int(lookback)
    # Keep at least one valid patch for short lookback values such as m=10.
    cfg.patch_len = int(max(1, min(cfg.patch_len, cfg.lookback)))
    return cfg


def run_e3tsf_only_for_input_length(dcfg: Dict, seed: int, lookback: int) -> Optional[Dict[str, float]]:
    dcfg_tmp = copy.deepcopy(dcfg)
    dcfg_tmp["config"] = _cfg_for_input_length(dcfg["config"], lookback)
    name = dcfg_tmp["name"]
    cfg = dcfg_tmp["config"]
    print("\n" + "-" * 100)
    print(f"Figure 8 experiment | dataset={name} | input length m={lookback} | horizon={cfg.horizon} | seed={seed}")
    try:
        set_seed(seed)
        train_loader, val_loader, test_loader, y_scaler, info = make_loaders(dcfg_tmp, seed=seed)
        model = E3TSF(info["n_scales"], cfg, n_series=info["n_series"]).to(DEVICE)
        sync_device()
        start = time.perf_counter()
        model = train_model(model, train_loader, val_loader, cfg)
        sync_device()
        train_seconds = time.perf_counter() - start
        sync_device()
        infer_start = time.perf_counter()
        y_true, y_pred, _ = predict_model(model, test_loader, y_scaler, return_gate=False)
        sync_device()
        inference_seconds = time.perf_counter() - infer_start
        metrics = compute_metrics(y_true, y_pred)
        del model
        if torch.cuda.is_available():
            torch.cuda.empty_cache()
        return {
            "dataset": name,
            "seed": seed,
            "input_length": int(lookback),
            "horizon": int(cfg.horizon),
            **metrics,
            "train_seconds": float(train_seconds),
            "inference_seconds": float(inference_seconds),
            "total_seconds": float(train_seconds + inference_seconds),
        }
    except Exception as e:
        print(f"Figure 8 experiment skipped for dataset={name}, m={lookback}: {repr(e)}")
        return None


def plot_figure8_input_length_metrics(fig8_df: pd.DataFrame):
    if fig8_df is not None and len(fig8_df) > 0:
        source_path = os.path.join(FIGURE_SOURCE_DIR, "Figure8_input_length_metrics_E3TSF_source.csv")
        fig8_df.to_csv(source_path, index=False)
        print(f"Saved Figure 8 source data: {source_path}")
    if not HAS_MPL or fig8_df is None or len(fig8_df) == 0:
        return
    metrics = ["RMSE", "RRSE", "CORR"]
    fig, axes = plt.subplots(3, 1, figsize=(8, 11), sharex=True)
    for ax, metric in zip(axes, metrics):
        for dataset, g in fig8_df.groupby("dataset", sort=False):
            g = g.sort_values("input_length")
            ax.plot(g["input_length"], g[metric], marker="o", label=dataset)
        ax.set_title(metric)
        ax.set_ylabel(metric)
        ax.grid(True, alpha=0.25)
        ax.legend()
    axes[-1].set_xlabel("Input length (m)")
    fig.suptitle("Evaluation metrics changes for each dataset according to the input length", y=0.995)
    fig.tight_layout()
    path = os.path.join(PAPER_FIG_DIR, "Figure8_input_length_metrics_E3TSF.png")
    fig.savefig(path, dpi=300, bbox_inches="tight")
    plt.close(fig)
    print(f"Saved Figure 8-style input-length graph: {path}")


def run_input_length_experiment(seed: int = SEED, input_lengths: List[int] = None) -> pd.DataFrame:
    if input_lengths is None:
        input_lengths = INPUT_LENGTHS_FOR_FIG8
    rows = []
    for dcfg in DATASETS:
        missing = [p for p in dataset_paths(dcfg) if not os.path.exists(p)]
        if missing:
            print(f"Skipping Figure 8 experiment for {dcfg['name']} because files are missing: {missing}")
            continue
        for m in input_lengths:
            row = run_e3tsf_only_for_input_length(dcfg, seed=seed, lookback=m)
            if row is not None:
                rows.append(row)
    fig8_df = pd.DataFrame(rows)
    if len(fig8_df) > 0:
        fig8_path = os.path.join(TABLE_DIR, "E3TSF_figure8_input_length_experiment.csv")
        fig8_df.to_csv(fig8_path, index=False)
        print(f"Saved Figure 8 input-length results: {fig8_path}")
        plot_figure8_input_length_metrics(fig8_df)
    return fig8_df

def main():
    print("E3-TSF Dynamic Gated Multi-Expert Transformer: RevIN + Sparse Gate + Multi-Seed Version")
    print(f"Output directory: {OUT_DIR}")
    print(f"Device: {DEVICE}")
    print(f"Seeds: {RUN_SEEDS}")
    auto_resolve_kaggle_input_paths()
    assert_datasets_available()
    save_run_metadata()

    all_results = []
    skipped = []
    for seed in RUN_SEEDS:
        print("\n" + "#" * 100)
        print(f"Starting full experiment for seed={seed}")
        print("#" * 100)
        for dcfg in DATASETS:
            missing = [p for p in dataset_paths(dcfg) if not os.path.exists(p)]
            if missing:
                print("\n" + "=" * 100)
                print(f"Skipping {dcfg['name']} because these files were not found:")
                for p in missing:
                    print("  -", p)
                skipped.append({"dataset": dcfg["name"], "seed": seed, "reason": "missing files", "paths": "; ".join(missing)})
                continue
            try:
                df_res = run_single_dataset(dcfg, seed=seed)
                all_results.append(df_res)
            except Exception as e:
                print(f"\nERROR while running {dcfg['name']} with seed={seed}: {repr(e)}")
                skipped.append({"dataset": dcfg["name"], "seed": seed, "reason": repr(e), "paths": ""})

    if all_results:
        summary = pd.concat(all_results, axis=0, ignore_index=True)
        summary_sorted = summary.sort_values(["dataset", "seed", "MAE"]).reset_index(drop=True)
        summary_path = os.path.join(TABLE_DIR, "E3TSF_all_datasets_all_seed_raw_summary.csv")
        summary_sorted.to_csv(summary_path, index=False)

        mean_std = summarize_mean_std(summary_sorted)
        mean_std_path = os.path.join(TABLE_DIR, "E3TSF_all_datasets_mean_std_summary.csv")
        mean_std.to_csv(mean_std_path, index=False)

        best_rows = mean_std.loc[mean_std.groupby("dataset")["MAE_mean"].idxmin()].reset_index(drop=True)
        best_path = os.path.join(TABLE_DIR, "E3TSF_best_model_per_dataset_mean_std.csv")
        best_rows.to_csv(best_path, index=False)

        # Win-count table for paper discussion.
        per_seed_best = summary_sorted.loc[summary_sorted.groupby(["dataset", "seed"])["MAE"].idxmin()].reset_index(drop=True)
        win_count = per_seed_best.groupby(["dataset", "model"]).size().reset_index(name="MAE_win_count")
        win_path = os.path.join(TABLE_DIR, "E3TSF_MAE_win_count_by_seed.csv")
        win_count.to_csv(win_path, index=False)
        save_overall_summary_plots(mean_std, best_rows, win_count)

        print("\n" + "=" * 100)
        print("Raw final summary sorted by dataset, seed, and MAE:")
        print(summary_sorted.to_string(index=False))
        print("\nMean/std summary sorted by dataset and MAE_mean:")
        print(mean_std.to_string(index=False))
        print(f"\nSaved raw summary: {summary_path}")
        print(f"Saved mean/std summary: {mean_std_path}")
        print(f"Saved best-model mean/std summary: {best_path}")
        print(f"Saved MAE win-count summary: {win_path}")
    else:
        print("\nNo datasets were executed. Please correct dataset paths in DATASETS.")

    if skipped:
        skipped_df = pd.DataFrame(skipped)
        skipped_path = os.path.join(TABLE_DIR, "E3TSF_skipped_or_failed_datasets.csv")
        skipped_df.to_csv(skipped_path, index=False)
        print(f"Saved skipped/failed dataset report: {skipped_path}")


    if RUN_INPUT_LENGTH_EXPERIMENT:
        print("\n" + "=" * 100)
        print("Running Figure 8-style input-length experiment for E3-TSF")
        print("=" * 100)
        run_input_length_experiment(seed=RUN_SEEDS[0], input_lengths=INPUT_LENGTHS_FOR_FIG8)


if __name__ == "__main__":
    main()
